# ASTRA · Notebook 01 — Replicating García-Jara et al. (2022): a conditional GAN for light-curve augmentation

**CS787 (Generative AI) project, Group 4.** Catalina runs on Tharun's Kaggle account, ZTF on Thamilselvan's.

**Goal of this notebook.** Reproduce Table 5 of the primary paper, as exactly as their text allows. `DATASET` picks the column: `"catalina"` (default) or `"ztf"`. The paper:
García-Jara, Protopapas & Estévez, *Improving Astronomical Time-series Classification via Data Augmentation with Generative Adversarial Networks*, ApJ 2022 (arXiv 2205.06758).

| Row | Training set of the classifier | Paper, Catalina (test accuracy %) | Paper, ZTF |
|---|---|---|---|
| A | real, balanced by oversampling (TRTR) | 73.44 ± 1.22 | 72.61 ± 0.69 |
| B / C / D | real + soft time / magnitude / mixed warping | 74.06 / 73.64 / 73.82 | 72.69 / 72.45 / 72.53 |
| E / F | u-GAN (balanced GAN training), chosen by val. accuracy / G-score (TSTR) | 75.97 / 76.28 | 74.17 / 73.79 |
| G / H | γ-GAN (γ = 0.25), chosen by val. accuracy / G-score (TSTR) | 76.86 / **76.97 ± 0.79** | 74.37 / **74.94 ± 0.44** |

TRTR = train on real, test on real. TSTR = train on synthetic *only*, test on real. Every row is 15 independent runs of an ensemble of 5 CNNs.

**The method, in the words of the GAN slides.** The **generator** turns noise z plus a condition z̄ = (observation phases, amplitude, class) into a light curve that should "look real". The **discriminator** gets real and generated ("fake") curves with the same condition and has two outputs: a real-vs-fake score, and a class prediction (AC-GAN). The paper uses the Wasserstein loss with a gradient penalty (WGAN-GP) instead of the slides' cross-entropy, for more stable training. The discriminator is trained 5 times per generator step, and when one network trains the other one's weights are held fixed.

The paper's own contributions that we replicate:
- **γ-resampling**: training batches are drawn with class probabilities between the natural mix (γ = 0) and fully balanced (γ = 1). Balanced batches repeat the rare classes so often that the GAN over-fits them quickly.
- **G-score**: picks a checkpoint from *density* (fidelity) and *recall* (diversity) in a classifier's feature space. It is ~6× cheaper than training classifiers on every checkpoint, and it catches over-fitting that FID misses.

**What is not in the paper and had to be chosen.** Each choice is marked "ours" in the code:
- GAN batch size 64 and gradient-penalty weight λ = 10 (both WGAN-GP defaults);
- the classifier's batch size (128) and dropout (0.5), and how long it trains: it is checked on the validation set 4 times per epoch and the best check is kept (early stopping, patience 3 epochs). We measured that it over-fits after about 2 epochs: test accuracy falls from 74% to 67% by epoch 10;
- **mixed precision (fp16)** for the GAN and the classifiers. In full precision a T4 needs about 24 h for one GAN run, and the paper does not say what precision or hardware it used;
- k = 5 for the k-NN balls;
- amplitude = half the peak-to-peak range of the 100 kept points;
- the width of the class-range outlier filter: ±2σ, which reproduces the paper's Table 2 class counts (see the data section);
- the soft-warping interval for k is read as [1/(2b), 2/b] (the paper prints *a*, which breaks for magnitudes);
- for ZTF: bands, dates and period source are not stated in the paper; they were recovered from its plotted light curves (one curve per star and band in g and r, the label file's catalogue period, detections up to 19 July 2020). The paper's number of curves per class is then drawn at random from those that qualify (see the data section).

**How to run**
- Accelerator **GPU T4 ×2** · Internet **On** · Persistence: none needed. For `DATASET = "ztf"`, also *Add Input* → the dataset that holds `ztf_L40.npz`.
- First run with `SMOKE = True` interactively (about 20 minutes). It runs the tests and a tiny version of every job, and times the real jobs (in full and in mixed precision) so their hours can be estimated.
- Then set `SMOKE = False` → **Save Version → Save & Run All**. GPU 0 and GPU 1 each work down their own queue of jobs (see `QUEUES`). Every job saves a resumable state at least every 30 minutes and stops cleanly before the 10.5-hour budget.
- **Next session:** *Add Input* → this notebook's previous version (its output), set `SESSION` one higher, Save & Run All again. Finished jobs are skipped, an unfinished GAN resumes from its last state.

In [ ]:
# ---- Configuration ----
import os, sys, json, time, shutil, subprocess
from pathlib import Path

SMOKE = True            # True: tests + tiny jobs + timing (run interactively first); False: the real run
DATASET = "catalina"    # "catalina", or "ztf" (then attach the dataset that holds ztf_L40.npz under Add Input)
SESSION = 1             # 1, 2, 3, ... only labels this run's summary
SESSION_HOURS = 10.5    # stop cleanly before Kaggle's 12-hour limit; lower it if the account has less quota left

# Each GPU works down its own queue; unfinished jobs resume in the next session (previous output attached).
# trtr = rows A-D, gan_u = u-GAN (rows E, F), gan_gamma025 = gamma-GAN (rows G, H); s0/s1/s2 = seeds
QUEUES = {
    "0": ["gan_gamma025_s0", "gan_gamma025_s1", "gan_gamma0_s0"],
    "1": ["trtr", "gan_u_s0", "gan_gamma025_s2"],
}
L, LATENT = {"catalina": (100, 16), "ztf": (40, 8)}[DATASET]    # points per curve, noise size (Sec. 3.2.3, 4.2)
CLASS_RANGE_K = 2.0     # class-range outlier filter width (not given in the paper; 2 sigma matches Table 2)
DATA_SEED = 0

LOCAL_TEST = os.environ.get("ASTRA_LOCAL_TEST") == "1"     # only my offline test run sets this
RUN_STARTED = time.time()
WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))
RUNS = WORK / (f"runs_{DATASET}" + ("_smoke" if SMOKE else ""))      # smoke outputs never mix with the real ones
FIGS, TABLES = WORK / f"figures_{DATASET}", WORK / f"tables_{DATASET}"
for d in (RUNS, FIGS, TABLES, WORK / "data", WORK / "astra"):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print(f"{DATASET}: session {SESSION} ({SESSION_HOURS} h), {'SMOKE' if SMOKE else 'REAL RUN'} | queues: {QUEUES}")

In [ ]:
# ---- GPUs, and restoring the previous session's output ----
import numpy as np, torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert LOCAL_TEST or torch.cuda.device_count() >= 2, "Set Accelerator to GPU T4 x2"
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def restore_previous():
    """Copies runs/, data/, figures/ and tables/ of an attached previous version, so finished jobs are
    skipped and unfinished ones resume. Files already present are kept."""
    n = 0
    for name in (f"runs_{DATASET}", "data", FIGS.name, TABLES.name):
        for root in INPUT.glob(f"*/**/{name}"):
            if not root.is_dir() or WORK in root.parents:
                continue
            for src in root.rglob("*"):
                dest = WORK / name / src.relative_to(root)
                if src.is_file() and not dest.exists():
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dest)
                    n += 1
    return n


n = restore_previous()
print(f"restored {n} file(s) from the previous session" if n else "fresh start (nothing to restore)")
for p in sorted(RUNS.glob("*/status.json")):
    print(f"  {p.parent.name:<18} {json.loads(p.read_text())}")

## The `astra` package

The code lives in small modules written to `/kaggle/working/astra/` by the next cells, so that the GPU workers (separate processes, one per GPU) can import it:
- `common`: paths, seeds, the time guard, safe saving;
- `catalina`, `ztf`: the two datasets (download and pre-processing);
- `sampling`: the γ-resampling block;
- `classifier`: the paper's CNN, trained as 75 independent copies at once;
- `warping`: the soft-warping baselines;
- `metrics`: density, recall, G-score, FID, Welch's test;
- `gan`: generator, discriminator, training loop, checkpoint evaluation and model selection;
- `run_gan`: the per-GPU worker.

In [ ]:
%%writefile astra/__init__.py
"""ASTRA: generative models for variable-star light curves (CS787 project, Group 4)."""


In [ ]:
%%writefile astra/common.py
"""Small helpers shared by every notebook: seeds, paths, time guard, safe saving, logging."""
import json
import os
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch

WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_json(obj, path):
    """Write to a temp file first, so a kill mid-write never leaves a broken file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def restore_rng(state, device):
    """Put back the random-number state saved in a resumable checkpoint. torch.load(map_location=GPU) moves
    the saved states to the GPU, but the generators only accept CPU tensors, so they are moved back first.
    Never fatal: without it a resumed run just continues with fresh random numbers. Returns True on success."""
    try:
        torch.set_rng_state(state["rng_cpu"].cpu())
        if state.get("rng_cuda") is not None and str(device).startswith("cuda"):
            torch.cuda.set_rng_state_all([s.cpu() for s in state["rng_cuda"]])
        return True
    except Exception as e:
        print(f"could not restore the random-number state ({type(e).__name__}: {e}); continuing with a fresh one",
              flush=True)
        return False


class Deadline:
    """Session time guard: Kaggle kills a session at 12 h, we stop cleanly before that."""

    def __init__(self, deadline_unix):
        self.t = float(deadline_unix)

    def left(self):
        return self.t - time.time()

    def passed(self, reserve_s=0.0):
        return self.left() < reserve_s


class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def __call__(self, *msg):
        line = time.strftime("%H:%M:%S ") + " ".join(str(m) for m in msg)
        print(line, flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def find_input(name):
    """A file attached under /kaggle/input (e.g. a previous version's output), if any."""
    hits = sorted(INPUT.glob(f"**/{name}")) if INPUT.exists() else []
    return hits[0] if hits else None


def download(url, dest, tries=5):
    """Download once; reuse a copy attached under /kaggle/input if there is one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    attached = find_input(dest.name)
    if attached is not None:
        return attached
    dest.parent.mkdir(parents=True, exist_ok=True)
    for i in range(tries):
        try:
            tmp = dest.with_suffix(dest.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, dest)
            return dest
        except Exception as e:  # network hiccup: wait and retry
            print(f"download failed ({e}), retry {i + 1}/{tries}")
            time.sleep(10 * (i + 1))
    raise RuntimeError(f"could not download {url}")


def peak_gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / 1e9


In [ ]:
%%writefile astra/catalina.py
"""Catalina Surveys DR1 periodic variables, pre-processed exactly as in Garcia-Jara et al. (2022), Sec. 3.

Steps: period folding -> outlier filtering (sliding z-score, then a class-range filter) -> keep curves with
at least L points -> random sub-sampling to L points -> median centering. Then the paper's split: validation
and test each hold 20% of the smallest class, for every class (Sec. 4.1).
"""
from pathlib import Path

import numpy as np
import pandas as pd

from .common import download

CATALOG_URL = "http://nesssi.cacr.caltech.edu/DataRelease/CatalinaVars.tbl"
PHOT_URL = "http://nesssi.cacr.caltech.edu/DataRelease/AllVar.phot.gz"

# Table 1 of the paper: Drake et al. (2014) type id -> new class
CLASS_MAP = {1: "EBSD/D", 3: "EBSD/D",                      # EW, beta Lyrae
             4: "RRL", 5: "RRL", 6: "RRL", 7: "RRL",        # RRab, RRc, RRd, Blazhko
             2: "EBC",                                      # EA
             13: "LPV",
             11: "DSCT", 12: "DSCT",                        # HADS, LADS
             9: "CEP", 10: "CEP"}                           # ACEP, Cep-II
CLASSES = ["EBSD/D", "RRL", "EBC", "LPV", "DSCT", "CEP"]
PAPER_COUNTS = {"EBSD/D": 28980, "RRL": 7533, "EBC": 4500, "LPV": 483, "DSCT": 241, "CEP": 182}   # Table 2
TRAIN, VAL, TEST = 0, 1, 2


def read_catalog(path):
    cols = ["css_id", "numid", "ra", "dec", "vmag", "period", "amp_cat", "nobs", "vartype"]
    cat = pd.read_csv(path, sep=r"\s+", comment="#", header=None, names=cols)
    cat = cat[cat.vartype.isin(list(CLASS_MAP))].copy()
    cat["period"] = pd.to_numeric(cat.period)          # "\nodata" only occurs in type 17, dropped above
    cat["cls"] = cat.vartype.map(CLASS_MAP)
    return cat.sort_values("numid").reset_index(drop=True)


def read_photometry(path, numids):
    """All (MJD, mag, err) rows of the wanted objects, grouped per object."""
    ph = pd.read_csv(path, header=None, usecols=[0, 1, 2, 3], names=["numid", "mjd", "mag", "err"],
                     dtype={"numid": np.int64, "mjd": np.float64, "mag": np.float64, "err": np.float64})
    ph = ph[ph.numid.isin(set(numids))].sort_values(["numid", "mjd"], kind="stable")
    ids, start = np.unique(ph.numid.values, return_index=True)
    end = np.append(start[1:], len(ph))
    t, m, e = ph.mjd.values, ph.mag.values, ph.err.values
    return {i: (t[a:b], m[a:b], e[a:b]) for i, a, b in zip(ids, start, end)}


def fold(t, period):
    """Eqs. 8-9: phase = (t mod T) / T, in [0, 1)."""
    return np.mod(t, period) / period


def zscore_keep(m, ws=20, z=3.0, passes=2):
    """Sliding-window z-score filter on a phase-sorted curve (window 20, drop z > 3, two passes)."""
    keep = np.ones(len(m), bool)
    for _ in range(passes):
        s = pd.Series(m[keep])
        mu = s.rolling(ws, center=True, min_periods=2).mean().values
        sd = s.rolling(ws, center=True, min_periods=2).std().values
        with np.errstate(invalid="ignore", divide="ignore"):
            bad = np.abs(s.values - mu) / sd > z
        idx = np.flatnonzero(keep)
        keep[idx[bad]] = False
    return keep


def preprocess(data_dir, L=100, class_range_k=3.0, seed=0, log=print):
    """Returns a dict of arrays (one row per kept star) plus a per-class count table."""
    data_dir = Path(data_dir)
    cat = read_catalog(download(CATALOG_URL, data_dir / "CatalinaVars.tbl"))
    log(f"catalog: {len(cat):,} stars in the 6 classes")
    phot = read_photometry(download(PHOT_URL, data_dir / "AllVar.phot.gz"), cat.numid.values)
    log(f"photometry found for {len(phot):,} of them")

    # 1-2. fold, sort by phase, sliding z-score filter
    curves = []
    for r in cat.itertuples():
        if r.numid not in phot:
            continue
        t, m, e = phot[r.numid]
        ph = fold(t, r.period)
        o = np.argsort(ph, kind="stable")
        ph, m, e = ph[o], m[o], e[o]
        k = zscore_keep(m)
        curves.append((r.numid, r.cls, r.period, ph[k], m[k], e[k]))

    # 2b. class-range filter: drop a curve if > 90% of its magnitudes lie outside median +- k*std of its class
    stats = {}
    for c in CLASSES:
        allm = np.concatenate([cv[4] for cv in curves if cv[1] == c])
        stats[c] = (np.median(allm), np.std(allm))
    kept, n_range, n_short = [], 0, 0
    for cv in curves:
        med, sd = stats[cv[1]]
        if np.mean(np.abs(cv[4] - med) > class_range_k * sd) > 0.9:
            n_range += 1
        elif len(cv[4]) < L:       # 3. need at least L points to sub-sample
            n_short += 1
        else:
            kept.append(cv)
    log(f"dropped {n_range} by the class-range filter and {n_short} with fewer than {L} points")

    # 4-5. random sub-sampling to L points (kept in phase order), then median centering
    rng = np.random.default_rng(seed)
    n = len(kept)
    out = dict(numid=np.zeros(n, np.int64), y=np.zeros(n, np.int64), period=np.zeros(n),
               phase=np.zeros((n, L), np.float32), mag=np.zeros((n, L), np.float32),
               err=np.zeros((n, L), np.float32), amp=np.zeros(n, np.float32))
    for i, (numid, c, period, ph, m, e) in enumerate(kept):
        s = np.sort(rng.choice(len(m), L, replace=False))
        x = m[s] - np.median(m[s])
        out["numid"][i], out["y"][i], out["period"][i] = numid, CLASSES.index(c), period
        out["phase"][i], out["mag"][i], out["err"][i] = ph[s], x, e[s]
        out["amp"][i] = (x.max() - x.min()) / 2          # half peak-to-peak of the L points

    # split (Sec. 4.1): val and test each get 20% of the smallest class, for every class
    counts = np.bincount(out["y"], minlength=len(CLASSES))
    n_hold = int(0.2 * counts.min())
    split = np.full(n, TRAIN, np.int8)
    for c in range(len(CLASSES)):
        idx = rng.permutation(np.flatnonzero(out["y"] == c))
        split[idx[:n_hold]] = VAL
        split[idx[n_hold:2 * n_hold]] = TEST
    out["split"] = split
    out["classes"] = np.array(CLASSES)
    table = pd.DataFrame({"class": CLASSES, "ours": counts,
                          "paper": [PAPER_COUNTS[c] for c in CLASSES]})
    table["ours/paper"] = (table.ours / table.paper).round(3)
    table["train"] = [int(((out["y"] == c) & (split == TRAIN)).sum()) for c in range(len(CLASSES))]
    table["val"] = n_hold
    table["test"] = n_hold
    return out, table


In [ ]:
%%writefile astra/ztf.py
"""The ZTF dataset of Garcia-Jara et al. (2022), rebuilt from public sources.

What the paper states (Sec. 3): the periodic variable stars of ALeRCE's ZTF training set (Sanchez-Saez et al.
2021; classes EB, RRL, LPV, DSCT, CEP), light curves "processed by the ALeRCE broker", anomalous observations
(marked with magnitude 100) removed with a threshold of 30, folded with the provided period, 40 randomly
chosen observations per curve, median-centred. Table 2 gives the class counts after pre-processing.

What it does not state: the photometric band, the date of the data, the period's origin, how stars were
selected. We recovered these from the paper itself (notes/ztf_exact_dataset_leads.md): the real light curves
plotted in the paper and in the first author's thesis were digitised and matched to ALeRCE's detections, which
identified the 21 stars shown. They tell us that
  - a light curve is the detections of one star in ONE band, and the dataset holds g curves and r curves
    (14 and 7 of the 21);
  - the period is the `period` column of ALeRCE's internal label file, i.e. the period of the catalogue the
    label came from (equal to 1e-5 for all 21), with phase = (MJD mod T) / T. Most Gaia-labelled stars have no
    period there (78% of the RR Lyrae, 74% of the LPVs), cannot be folded, and drop out;
  - no plotted point is later than MJD 59048.3 (2020-07-19), and many are that late: one snapshot of that date;
  - only the magnitude threshold is applied; the 40 points are centred on their own median.
With one curve per (star, band), stars that have a period, detections up to that date and repeated epochs
dropped, "at least 40 points" gives EB 31,174 / RRL 18,748 / LPV 5,470, against Table 2's 31,477 / 18,729 /
5,245. DSCT and CEP come out at 768 and 716 against 507 and 471, which we cannot explain. So:
  - light curves = ALeRCE detections (TAP table ztf.detection), corrected magnitudes magpsf_corr < 30, per band;
  - keep curves with at least L points, then draw at random exactly the paper's number of curves per class
    (all of them where fewer qualify: EB), so the class imbalance and the split sizes match Table 2;
  - split as in Sec. 4.1: validation and test each hold 20% of the smallest class, for every class. As in the
    paper the split is over curves, so the g and the r curve of a star can fall on different sides.
"""
import numpy as np
import pandas as pd

CLASSES = ["EB", "RRL", "LPV", "DSCT", "CEP"]                    # the paper's names, in Table 2 order
ALERCE_NAME = {"EB": "E", "RRL": "RRL", "LPV": "LPV", "DSCT": "DSCT", "CEP": "CEP"}
PAPER_COUNTS = {"EB": 31477, "RRL": 18729, "LPV": 5245, "DSCT": 507, "CEP": 471}
TRAIN, VAL, TEST = 0, 1, 2
MAG_THR = 30.0
MJD_MAX = 59048.4                                                 # 2020-07-19
BANDS = (1, 2)                                                    # g, r


def preprocess(raw_npz, L=40, bands=BANDS, seed=0, match_paper_counts=True, period_key="period_label",
               mjd_max=MJD_MAX, log=print):
    """raw_npz: the file written by build/ztf/download_ztf.py and add_label_periods.py. bands: 1 = g, 2 = r.
    period_key: "period_label" (the label file's catalogue period) or "period" (ALeRCE's Multiband_period,
    which is half the orbital period for most eclipsing binaries).
    mjd_max: the last date of the detections used (None = all of them)."""
    z = np.load(raw_npz)
    period, cls = z[period_key], z["cls"]
    ok = np.isin(z["fid"], bands) & (z["mag"] < MAG_THR)
    if mjd_max is not None:
        ok &= z["mjd"] <= mjd_max
    # one light curve per (star, band): sort its detections by time and drop repeated epochs
    cur, mjd, mag, err = z["obj"][ok].astype(np.int64) * 4 + z["fid"][ok], z["mjd"][ok], z["mag"][ok], z["err_ext"][ok]
    order = np.lexsort((mjd, cur))
    cur, mjd, mag, err = cur[order], mjd[order], mag[order], err[order]
    new = np.r_[True, (cur[1:] != cur[:-1]) | (mjd[1:] != mjd[:-1])]
    cur, mjd, mag, err = cur[new], mjd[new], mag[new], err[new]
    ids, start, cnt = np.unique(cur, return_index=True, return_counts=True)
    star, band = ids // 4, ids % 4
    good = (cnt >= L) & np.isfinite(period[star]) & (period[star] > 0)
    rng = np.random.default_rng(seed)
    chosen, qualify = [], {}
    for c in CLASSES:
        idx = np.flatnonzero(good & (cls[star] == ALERCE_NAME[c]))
        qualify[c] = len(idx)
        k = min(PAPER_COUNTS[c], len(idx)) if match_paper_counts else len(idx)
        chosen.append(np.sort(rng.choice(idx, k, replace=False)))
    y = np.concatenate([np.full(len(v), i) for i, v in enumerate(chosen)])
    chosen = np.concatenate(chosen)
    n = len(chosen)
    out = dict(oid=z["oid"][star[chosen]], band=band[chosen].astype(np.int8), y=y, period=period[star[chosen]],
               phase=np.zeros((n, L), np.float32), mag=np.zeros((n, L), np.float32),
               err=np.zeros((n, L), np.float32), amp=np.zeros(n, np.float32))
    for i, k in enumerate(chosen):
        s = rng.choice(cnt[k], L, replace=False) + start[k]       # L random observations
        ph = np.mod(mjd[s], out["period"][i]) / out["period"][i]  # Eqs. 8-9
        o = np.argsort(ph, kind="stable")
        x = mag[s][o] - np.median(mag[s])                         # median centring
        out["phase"][i], out["mag"][i], out["err"][i] = ph[o], x, err[s][o]
        out["amp"][i] = (x.max() - x.min()) / 2
    counts = np.bincount(y, minlength=len(CLASSES))
    n_hold = int(0.2 * counts.min())
    split = np.full(n, TRAIN, np.int8)
    for c in range(len(CLASSES)):
        idx = rng.permutation(np.flatnonzero(y == c))
        split[idx[:n_hold]] = VAL
        split[idx[n_hold:2 * n_hold]] = TEST
    out["split"] = split
    out["classes"] = np.array(CLASSES)
    has_p = np.isfinite(period) & (period > 0)
    table = pd.DataFrame({"class": CLASSES, "labeled stars": [int(np.sum(cls == ALERCE_NAME[c])) for c in CLASSES],
                          "with a period": [int(np.sum((cls == ALERCE_NAME[c]) & has_p)) for c in CLASSES],
                          f"curves >= {L} points": [qualify[c] for c in CLASSES], "ours": counts,
                          "paper": [PAPER_COUNTS[c] for c in CLASSES]})
    for b, name in ((1, "g"), (2, "r")):
        table[name] = [int(((y == c) & (out["band"] == b)).sum()) for c in range(len(CLASSES))]
    table["train"] = [int(((y == c) & (split == TRAIN)).sum()) for c in range(len(CLASSES))]
    table["val"] = n_hold
    table["test"] = n_hold
    log(f"{n:,} light curves of {L} points ({int((out['band'] == 1).sum()):,} in g, {int((out['band'] == 2).sum()):,} in r)")
    return out, table


In [ ]:
%%writefile astra/sampling.py
"""The resampling block of Garcia-Jara et al. (2022), Sec. 4.6.

Each class keeps its own shuffled queue. A class is picked with probability
    p_i = gamma / N + (1 - gamma) * S_i / S          (Eq. 12)
and the next sample is taken from its queue; an empty queue is reshuffled.
gamma = 0 keeps the original class mix, gamma = 1 gives a uniformly balanced stream.
"""
import numpy as np


class ClassResampler:
    def __init__(self, y, gamma, seed=0, n_classes=None):
        y = np.asarray(y)
        self.n_classes = n_classes or int(y.max()) + 1
        self.rng = np.random.default_rng(seed)
        self.pools = [np.flatnonzero(y == c) for c in range(self.n_classes)]
        sizes = np.array([len(p) for p in self.pools], float)
        self.p = gamma / self.n_classes + (1 - gamma) * sizes / sizes.sum()
        self.p[sizes == 0] = 0
        self.p /= self.p.sum()
        self.queues = [self.rng.permutation(p) for p in self.pools]
        self.pos = np.zeros(self.n_classes, int)

    def _take(self, c, k):
        out = []
        while k > 0:
            if self.pos[c] == len(self.queues[c]):
                self.queues[c] = self.rng.permutation(self.pools[c])
                self.pos[c] = 0
            n = min(k, len(self.queues[c]) - self.pos[c])
            out.append(self.queues[c][self.pos[c]:self.pos[c] + n])
            self.pos[c] += n
            k -= n
        return np.concatenate(out)

    def sample(self, batch):
        """Indices of one batch (classes drawn with p, samples without replacement inside a class)."""
        cls = self.rng.choice(self.n_classes, size=batch, p=self.p)
        idx = np.empty(batch, np.int64)
        for c in np.unique(cls):
            where = np.flatnonzero(cls == c)
            idx[where] = self._take(c, len(where))
        return idx


In [ ]:
%%writefile astra/classifier.py
"""The downstream classifier of Garcia-Jara et al. (2022), Sec. 4.3 and Table 3.

Base classifier: 5 conv blocks (Conv k=3 + BatchNorm, MaxPool 2 + ReLU), channels 2->32->64->128->64->64,
then Dense -> 100, BN + ReLU + Dropout, Dense -> N. Input = magnitudes and phases (2 channels).
The classifier is an ensemble of 5 base CNNs, and every score is repeated over 15 independent runs,
i.e. 75 small CNNs per table row. We train them side by side as ONE network with a leading "member"
axis: every member has its own weights and its own mini-batches, so they are exactly as independent
as 75 separate trainings. Each layer is one batched matrix product over the members (a convolution
with kernel 3 is a matrix product on the 3 shifted copies of its input), which a GPU runs far faster
than 75 small networks or a grouped convolution (measured on a T4: 133 s per epoch with grouped
convolutions). Mixed precision (fp16) is used on the GPU for the same reason.
"""
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .sampling import ClassResampler


class MemberLinear(nn.Module):
    """M independent Linear layers. x: (M, B, fin) -> (M, B, fout)."""

    def __init__(self, M, fin, fout):
        super().__init__()
        bound = 1 / math.sqrt(fin)                     # same init as nn.Linear
        self.w = nn.Parameter(torch.empty(M, fin, fout).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(M, 1, fout).uniform_(-bound, bound))

    def forward(self, x):
        return torch.baddbmm(self.b, x, self.w)


class MemberConv(nn.Module):
    """M independent Conv1d(kernel 3, padding 1) layers. x: (M, cin, B, L) -> (M, cout, B, L)."""

    def __init__(self, M, cin, cout, k=3):
        super().__init__()
        self.k = k
        bound = 1 / math.sqrt(cin * k)                 # same init as nn.Conv1d
        self.w = nn.Parameter(torch.empty(M, cout, cin * k).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(M, cout, 1).uniform_(-bound, bound))

    def forward(self, x):
        M, C, B, L = x.shape
        xp = F.pad(x, (self.k // 2, self.k // 2))
        cols = torch.stack([xp[..., i:i + L] for i in range(self.k)], 2)       # (M, cin, k, B, L)
        return torch.baddbmm(self.b, self.w, cols.reshape(M, C * self.k, B * L)).view(M, -1, B, L)


class CNNEnsemble(nn.Module):
    """M copies of the paper's base classifier (Table 3), run side by side. Input (M, B, in_ch, L)."""

    def __init__(self, M, n_classes, L, in_ch=2, dropout=0.5):
        super().__init__()
        self.M, self.in_ch = M, in_ch
        chans = [in_ch, 32, 64, 128, 64, 64]
        self.convs = nn.ModuleList(MemberConv(M, a, b) for a, b in zip(chans[:-1], chans[1:]))
        self.bns = nn.ModuleList(nn.BatchNorm2d(M * b) for b in chans[1:])
        l = L
        for _ in range(5):
            l = math.ceil(l / 2)                       # ceil(L/32) as in Table 3
        self.fc1 = MemberLinear(M, 64 * l, 100)
        self.bn = nn.BatchNorm1d(M * 100)
        self.drop = nn.Dropout(dropout)
        self.fc2 = MemberLinear(M, 100, n_classes)

    def features(self, x):
        """Output of the last conv block, flattened: (M, B, 64 * ceil(L/32))."""
        h = x.transpose(1, 2)                                              # (M, C, B, L)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h)
            M, C, B, L = h.shape
            h = bn(h.reshape(1, M * C, B, L))                              # statistics over (B, L) per member-channel
            h = F.relu(F.max_pool2d(h, (1, 2), ceil_mode=True)).view(M, C, B, -1)
        return h.transpose(1, 2).reshape(h.shape[0], h.shape[2], -1)

    def forward(self, x):
        h = self.fc1(self.features(x))                                     # (M, B, 100)
        M, B, _ = h.shape
        h = self.bn(h.transpose(1, 2).reshape(1, M * 100, B)).view(M, 100, B).transpose(1, 2)
        return self.fc2(self.drop(F.relu(h)))                              # (M, B, n_classes) logits


def reference_cnn(ens, m):
    """Member m of an ensemble rebuilt from standard PyTorch layers (used only by the self-tests)."""
    layers = []
    for conv in ens.convs:
        cout, cin = conv.w.shape[1], conv.w.shape[2] // conv.k
        ref = nn.Conv1d(cin, cout, conv.k, padding=conv.k // 2)
        ref.weight.data, ref.bias.data = conv.w[m].reshape(cout, cin, conv.k).clone(), conv.b[m, :, 0].clone()
        layers += [ref, nn.BatchNorm1d(cout), nn.MaxPool1d(2, ceil_mode=True), nn.ReLU()]
    fcs = []
    for fc in (ens.fc1, ens.fc2):
        ref = nn.Linear(fc.w.shape[1], fc.w.shape[2])
        ref.weight.data, ref.bias.data = fc.w[m].T.clone(), fc.b[m, 0].clone()
        fcs.append(ref)
    return nn.Sequential(*layers, nn.Flatten(), fcs[0], nn.BatchNorm1d(100), nn.ReLU(), fcs[1])


def stack_input(x, ph):
    """(n, L) magnitudes + (n, L) phases -> (n, 2, L)."""
    return torch.stack([x, ph], dim=1)


def amp_on(amp, device):
    return bool(amp) and str(device).startswith("cuda")


def autocast(device, amp):
    """fp16 mixed precision on the GPU when amp is on; a no-op otherwise (e.g. in the offline CPU tests)."""
    on = amp_on(amp, device)
    return torch.autocast("cuda" if on else "cpu", enabled=on)


@torch.no_grad()
def predict_proba(model, X, batch=512, amp=True):
    """Softmax outputs of every member: (M, n, N). X: (n, 2, L) on the model's device."""
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].unsqueeze(0).expand(model.M, -1, -1, -1)
        with autocast(X.device.type, amp):
            logits = model(xb)
        out.append(F.softmax(logits.float(), -1))
    return torch.cat(out, 1)


def macro_f1(y, pred, n_classes):
    f1 = []
    for c in range(n_classes):
        tp = np.sum((pred == c) & (y == c))
        p = tp / max(np.sum(pred == c), 1)
        r = tp / max(np.sum(y == c), 1)
        f1.append(0.0 if p + r == 0 else 2 * p * r / (p + r))
    return float(np.mean(f1))


def fit_ensembles(train, val, test, n_classes, runs=15, ens=5, max_epochs=10, patience=3, batch=128,
                  lr=1e-4, seed=0, warp=None, device="cuda", log=print, tag="", amp=True, checks_per_epoch=4):
    """Train `runs` independent ensembles of `ens` CNNs and score each run on val and test.

    train/val/test: dicts of torch tensors x (n, L), ph (n, L), y (n,). Training batches are class-balanced
    (the resampling block with gamma = 1, one independent stream per member), one epoch = len(train) samples.
    Each run is scored on val and test 4 times per epoch; we keep the check with the lowest ensemble validation
    loss (early stopping) and report its test scores. Measured on Catalina: the validation loss is lowest
    after ~2 epochs and test accuracy then falls from 74% to 67% by epoch 10, because the balanced batches
    repeat the rarest class ~66 times per epoch. So training stops once no run improved for `patience` epochs.
    `warp` optionally augments every training batch (the soft-warping baselines).
    """
    torch.manual_seed(seed)
    M = runs * ens
    L = train["x"].shape[1]
    model = CNNEnsemble(M, n_classes, L).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler(enabled=amp_on(amp, device))
    Xtr = stack_input(train["x"], train["ph"]).to(device)
    ytr = train["y"].to(device)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].cpu().numpy()
    Xte, yte = stack_input(test["x"], test["ph"]).to(device), test["y"].cpu().numpy()
    y_np = train["y"].cpu().numpy()
    samplers = [ClassResampler(y_np, gamma=1.0, seed=seed * 1000 + m, n_classes=n_classes) for m in range(M)]
    steps = math.ceil(len(y_np) / batch / checks_per_epoch)            # training steps between two checks

    best_loss = np.full(runs, np.inf)
    best = {"epoch": np.zeros(runs), "val_acc": np.zeros(runs), "test_acc": np.zeros(runs),
            "test_f1": np.zeros(runs), "test_pred": np.zeros((runs, len(yte)), int)}
    curves = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "test_acc": []}
    since = np.zeros(runs, int)
    for check in range(1, int(max_epochs * checks_per_epoch) + 1):
        epoch = check / checks_per_epoch
        model.train()
        idx = torch.from_numpy(np.stack([s.sample(steps * batch) for s in samplers])).to(device)   # (M, S*B)
        tot = torch.zeros((), device=device)                            # summed on the GPU (no sync per step)
        for s in range(steps):
            ib = idx[:, s * batch:(s + 1) * batch]                    # (M, B)
            xb, yb = Xtr[ib], ytr[ib]                                  # (M, B, 2, L), (M, B)
            if warp is not None:
                xb = warp(xb)
            with autocast(device, amp):
                logits = model(xb)
            loss = F.cross_entropy(logits.float().reshape(-1, n_classes), yb.reshape(-1)) * M   # sum over members
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.detach() / M
        tot = tot.item()
        # scores of every run (= average of its members' softmax) on val and test: (runs, n, N)
        pv = predict_proba(model, Xva, amp=amp).reshape(runs, ens, len(yva), n_classes).mean(1).cpu().numpy()
        pt = predict_proba(model, Xte, amp=amp).reshape(runs, ens, len(yte), n_classes).mean(1).cpu().numpy()
        vloss = -np.log(np.take_along_axis(pv, yva[None, :, None], 2)[:, :, 0] + 1e-12).mean(1)   # (runs,)
        vacc = (pv.argmax(2) == yva[None]).mean(1)
        tpred = pt.argmax(2)                                                                    # (runs, n)
        tacc = (tpred == yte[None]).mean(1)
        for r in range(runs):
            if vloss[r] < best_loss[r]:
                best_loss[r], since[r] = vloss[r], 0
                best["epoch"][r], best["val_acc"][r], best["test_acc"][r] = epoch, vacc[r], tacc[r]
                best["test_f1"][r] = macro_f1(yte, tpred[r], n_classes)
                best["test_pred"][r] = tpred[r]
            else:
                since[r] += 1
        curves["epoch"].append(epoch)
        curves["train_loss"].append(tot / steps)
        curves["val_loss"].append(float(vloss.mean()))
        curves["val_acc"].append(float(vacc.mean()))
        curves["test_acc"].append(float(tacc.mean()))
        if check % checks_per_epoch == 0:
            log(f"  {tag} epoch {epoch:.0f}: train loss {tot / steps:.3f} | val loss {vloss.mean():.3f} "
                f"acc {vacc.mean():.3f} | test acc {tacc.mean():.3f}")
        if (since >= patience * checks_per_epoch).all():
            break
    recall = np.array([[np.mean(best["test_pred"][r][yte == c] == c) for c in range(n_classes)]
                       for r in range(runs)])
    return {"test_acc": best["test_acc"].tolist(), "test_f1": best["test_f1"].tolist(),
            "val_acc": best["val_acc"].tolist(), "best_epoch": best["epoch"].tolist(),
            "test_recall_per_class": recall.tolist(), "epochs_run": epoch, "curves": curves}


def train_feature_cnn(train, val, n_classes, max_epochs=20, patience=4, batch=128, lr=1e-4, seed=0,
                      device="cuda", log=print):
    """One base classifier trained on the balanced real training set (D_train^u). Its last conv block is the
    feature space for density, recall, FID and the G-score (Sec. 4.3). Keeps the lowest-val-loss weights.
    (Full precision: it is a single small network and its features are reused by every metric.)"""
    torch.manual_seed(seed)
    L = train["x"].shape[1]
    model = CNNEnsemble(1, n_classes, L).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    Xtr, ytr = stack_input(train["x"], train["ph"]).to(device), train["y"].to(device)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].to(device)
    sampler = ClassResampler(train["y"].cpu().numpy(), gamma=1.0, seed=seed, n_classes=n_classes)
    steps = math.ceil(len(ytr) / batch)
    best, best_state, since, curve = np.inf, None, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        idx = torch.from_numpy(sampler.sample(steps * batch)).to(device)
        tot = torch.zeros((), device=device)
        for s in range(steps):
            ib = idx[s * batch:(s + 1) * batch]
            loss = F.cross_entropy(model(Xtr[ib].unsqueeze(0))[0], ytr[ib])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.detach()
        p = predict_proba(model, Xva, amp=False)[0]
        vloss = F.nll_loss(torch.log(p + 1e-12), yva).item()
        vacc = (p.argmax(1) == yva).float().mean().item()
        curve.append((tot.item() / steps, vloss, vacc))
        if vloss < best:
            best, since = vloss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
        if since >= patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    log(f"feature CNN: {epoch} epochs, best val loss {best:.3f}, val acc at best "
        f"{[c[2] for c in curve][int(np.argmin([c[1] for c in curve]))]:.3f}")
    return model, curve


@torch.no_grad()
def extract_features(model, x, ph, batch=1024):
    """Feature vectors (n, 64 * ceil(L/32)) of light curves x, ph (torch tensors on the model's device)."""
    model.eval()
    out = []
    for i in range(0, len(x), batch):
        out.append(model.features(stack_input(x[i:i + batch], ph[i:i + batch]).unsqueeze(0))[0])
    return torch.cat(out)


In [ ]:
%%writefile astra/warping.py
"""Soft window-warping baselines of Garcia-Jara et al. (2022), Sec. 4.8.3, Eqs. 17-18.

A random phase window [t1, t2] is picked; values v inside it are mapped by f(v) = a + b*tanh(k*(v - c)),
with a = c = window centre and b = half-width, so the window never grows (f(lo) >= lo, f(hi) <= hi).
k ~ U[1/(2b), 2/b]: small k*b contracts towards the centre, large k*b (saturated tanh) expands.
(The paper prints the interval with `a`; with `a` the magnitude version breaks, because the centre of
median-centred magnitudes can be 0 or negative, so we read it as the half-width b.)
Time-warping maps the phases, magnitude-warping maps the magnitudes (window limits m1, m2 = min/max
magnitude inside the phase window), mixed-warping does both.
"""
import torch


def _tanh_map(v, lo, hi, inside):
    a = (lo + hi) / 2
    b = (hi - lo) / 2
    ok = inside & (b > 0)
    b_safe = torch.where(b > 0, b, torch.ones_like(b))
    k = (0.5 + 1.5 * torch.rand_like(b)) / b_safe          # U[1/(2b), 2/b]
    return torch.where(ok, a + b * torch.tanh(k * (v - a)), v)


def soft_warp(xb, mode):
    """xb: (..., 2, L) with channel 0 = magnitudes, channel 1 = phases. Returns a warped copy."""
    x, ph = xb[..., 0, :], xb[..., 1, :]
    shape = x.shape[:-1] + (1,)
    t = torch.sort(torch.rand(shape[:-1] + (2,), device=xb.device), dim=-1).values
    t1, t2 = t[..., :1], t[..., 1:]
    inside = (ph >= t1) & (ph <= t2)
    if mode in ("time", "mixed"):
        ph = _tanh_map(ph, t1, t2, inside)
    if mode in ("mag", "mixed"):
        big = torch.finfo(x.dtype).max
        m1 = torch.where(inside, x, torch.full_like(x, big)).min(-1, keepdim=True).values
        m2 = torch.where(inside, x, torch.full_like(x, -big)).max(-1, keepdim=True).values
        enough = inside.sum(-1, keepdim=True) >= 2
        m1 = torch.where(enough, m1, torch.zeros_like(m1))
        m2 = torch.where(enough, m2, torch.zeros_like(m2))
        x = _tanh_map(x, m1, m2, inside & enough)
    return torch.stack([x, ph], dim=-2)


def make_warp(mode):
    return None if mode is None else (lambda xb: soft_warp(xb, mode))


In [ ]:
%%writefile astra/metrics.py
"""Feature-space metrics used by Garcia-Jara et al. (2022): density & recall (Eqs. 5-6), the G-score
(Eqs. 13-16) and FID (Eq. 3). Precision and coverage are computed too, for completeness.

A ball B(phi, r) is the sphere around phi whose radius r is the distance to its k-th nearest neighbour
inside the same set (Eq. 4).
"""
import numpy as np
import torch
from scipy import linalg, stats


def knn_radii(f, k, chunk=4096):
    """Distance from every row of f to its k-th nearest neighbour in f (itself excluded)."""
    out = []
    for i in range(0, len(f), chunk):
        d = torch.cdist(f[i:i + chunk], f)
        out.append(d.topk(k + 1, dim=1, largest=False).values[:, k])      # column 0 is the point itself
    return torch.cat(out)


def prdc(real, fake, k=5, real_radii=None, chunk=4096):
    """Precision, recall, density, coverage of fake features against real features (torch, same device)."""
    rr = knn_radii(real, k) if real_radii is None else real_radii
    rf = knn_radii(fake, k)
    in_real_balls = torch.zeros(len(fake), device=real.device)       # how many real balls hold each fake
    real_covered = torch.zeros(len(real), dtype=torch.bool, device=real.device)
    real_recalled = torch.zeros(len(real), dtype=torch.bool, device=real.device)
    for i in range(0, len(real), chunk):
        d = torch.cdist(real[i:i + chunk], fake)                     # (chunk, n_fake)
        inside = d <= rr[i:i + chunk, None]
        in_real_balls += inside.sum(0)
        real_covered[i:i + chunk] = inside.any(1)
        real_recalled[i:i + chunk] = (d <= rf[None, :]).any(1)
    return {"precision": (in_real_balls > 0).float().mean().item(),
            "recall": real_recalled.float().mean().item(),
            "density": (in_real_balls.sum() / (k * len(fake))).item(),
            "coverage": real_covered.float().mean().item()}


def g_score(D, R):
    """D, R: arrays (n_checkpoints, n_classes). Per-class min-max normalisation over the checkpoints of one
    run (Eqs. 13-14), per-class F-score (Eq. 15), macro average (Eq. 16). Returns one score per checkpoint."""
    D, R = np.asarray(D, float), np.asarray(R, float)

    def mm(a):
        lo, hi = a.min(0, keepdims=True), a.max(0, keepdims=True)
        return (a - lo) / np.where(hi > lo, hi - lo, 1.0)

    Dn, Rn = mm(D), mm(R)
    F = np.where(Dn + Rn > 0, 2 * Dn * Rn / np.where(Dn + Rn > 0, Dn + Rn, 1.0), 0.0)
    return F.mean(1)


def gauss_stats(f):
    """Mean and covariance of a feature set (numpy, float64)."""
    return f.mean(0), np.cov(f, rowvar=False)


def fid_from_stats(mu1, s1, mu2, s2):
    """Frechet distance between two Gaussians (Eq. 3)."""
    covmean = linalg.sqrtm(s1 @ s2)
    if not np.isfinite(covmean).all():                                 # standard fix for a singular product
        eps = np.eye(len(s1)) * 1e-6
        covmean = linalg.sqrtm((s1 + eps) @ (s2 + eps))
    return float(np.sum((mu1 - mu2) ** 2) + np.trace(s1 + s2 - 2 * np.real(covmean)))


def fid(real, fake):
    """Frechet distance between Gaussians fitted to two feature sets (numpy arrays)."""
    return fid_from_stats(*gauss_stats(real), *gauss_stats(fake))


def welch_p(a, b):
    """Two-sided Welch's t-test p-value, as in Table 5."""
    return float(stats.ttest_ind(a, b, equal_var=False).pvalue)


In [ ]:
%%writefile astra/gan.py
"""The conditional WGAN-GP of Garcia-Jara et al. (2022), Sec. 4.2-4.7 and Table 4.

Generator G(z, zbar) with zbar = [phases (L), amplitude (1), one-hot class (N)]: Dense+ReLU -> 4 x 1024,
five transposed-conv blocks (stride 2, kernel 5) 1024->512->256->128->64->1, output tanh * s.
Discriminator D(x, zbar-tensor): five conv blocks (stride 2, kernel 5, LeakyReLU) 64->128->256->512->1024,
Dense -> N + 1 outputs: D_rg (real vs fake score) and D_y (class logits, the AC-GAN part).

Losses (Eqs. 1, 2, 10, 11), minimised:
  L_D = E[D_rg(x_g)] - E[D_rg(x_r)] + lambda * GP + xi * (H_r + H_g) + eps * E[D_rg(x_r)^2]
  L_G = -E[D_rg(x_g)] + xi * H_g
with n_disc = 5 D steps per G step, Adam(1e-4, 0.5, 0.9), 400k G iterations, EMA(0.999) of G's weights.

Training uses mixed precision (fp16 on the GPU, cfg["amp"]): in full precision a T4 does ~4.6 iterations/s,
i.e. 24 h for one run. The gradient penalty follows PyTorch's AMP recipe (scale the output before
autograd.grad, un-scale the gradient). Generation and all metrics stay in full precision.
"""
import copy
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .classifier import amp_on, autocast, extract_features, fit_ensembles
from .common import Logger, load_json, restore_rng, save_json, torch_save
from .metrics import fid_from_stats, g_score, gauss_stats, knn_radii, prdc
from .sampling import ClassResampler

TRAIN, VAL, TEST = 0, 1, 2
GP_SCALE_MAX = 1024.0


class Generator(nn.Module):
    def __init__(self, latent, L, n_classes, scale):
        super().__init__()
        self.L, self.n_classes, self.latent = L, n_classes, latent
        self.register_buffer("scale", torch.tensor(float(scale)))
        self.fc = nn.Linear(latent + L + 1 + n_classes, 4 * 1024)
        chans = [1024, 512, 256, 128, 64, 1]
        self.deconv = nn.ModuleList(
            [nn.ConvTranspose1d(cin, cout, 5, stride=2, padding=2, output_padding=1)
             for cin, cout in zip(chans[:-1], chans[1:])])

    def forward(self, z, ph, amp, onehot):
        h = F.relu(self.fc(torch.cat([z, ph, amp[:, None], onehot], 1))).view(-1, 1024, 4)
        for i, layer in enumerate(self.deconv):
            h = layer(h)
            if i < len(self.deconv) - 1:
                h = F.relu(h)
        return (torch.tanh(h) * self.scale)[:, 0, :self.L]     # length 4 * 2^5 = 128, keep the first L


class Discriminator(nn.Module):
    def __init__(self, L, n_classes):
        super().__init__()
        chans = [1 + 2 + n_classes, 64, 128, 256, 512, 1024]
        self.conv = nn.ModuleList([nn.Conv1d(cin, cout, 5, stride=2, padding=2)
                                   for cin, cout in zip(chans[:-1], chans[1:])])
        l = L
        for _ in range(5):
            l = math.ceil(l / 2)
        self.fc = nn.Linear(1024 * l, n_classes + 1)

    def forward(self, x, ph, amp, onehot):
        L = x.shape[1]
        cond = torch.cat([ph[:, None], amp[:, None, None].expand(-1, 1, L), onehot[:, :, None].expand(-1, -1, L)], 1)
        h = torch.cat([x[:, None], cond], 1)
        for layer in self.conv:
            h = F.leaky_relu(layer(h), 0.2)
        out = self.fc(h.flatten(1))
        return out[:, 0], out[:, 1:]                           # D_rg, D_y logits


def gradient_penalty(D, x_r, x_g, cond, scaler=None, amp=False):
    eps = torch.rand(len(x_r), 1, device=x_r.device)
    x_hat = (eps * x_r + (1 - eps) * x_g).requires_grad_(True)
    with autocast(x_r.device.type, amp):
        d_hat, _ = D(x_hat, *cond)
    # AMP recipe: scale the output so small gradients do not underflow in fp16, then divide the scale out again.
    # The scale is capped: the scaler starts at 65,536, and that already overflows fp16 (max 65,504) here,
    # because the gradient of the sum with respect to every critic output is exactly the scale.
    scale = min(scaler.get_scale(), GP_SCALE_MAX) if scaler is not None and scaler.is_enabled() else 1.0
    grad, = torch.autograd.grad(d_hat.float().sum() * scale, x_hat, create_graph=True)
    return (((grad / scale).norm(2, dim=1) - 1) ** 2).mean()


@torch.no_grad()
def generate(G, data, idx, seed, batch=4096):
    """One synthetic curve per index: the conditions (phases, amplitude, class) of real training star idx[i]."""
    g = torch.Generator(device=data["x"].device).manual_seed(seed)
    out = []
    for i in range(0, len(idx), batch):
        j = idx[i:i + batch]
        z = torch.randn(len(j), G.latent, device=data["x"].device, generator=g)
        out.append(G(z, data["ph"][j], data["amp"][j], data["onehot"][j]))
    return torch.cat(out)


def balanced_idx(data, n, seed, gamma=1.0):
    """n training-set indices from the resampling block (gamma = 1: balanced, i.e. D_train^u)."""
    rs = ClassResampler(data["y_np"][data["train_idx_np"]], gamma, seed=seed, n_classes=data["n_classes"])
    return data["train_idx"][torch.from_numpy(rs.sample(n)).to(data["x"].device)]


def load_data(npz_path, device):
    z = np.load(npz_path)
    n_classes = int(z["y"].max()) + 1
    d = {k: torch.from_numpy(z[k]).to(device) for k in ("phase", "mag", "amp", "y")}
    data = {"x": d["mag"].float(), "ph": d["phase"].float(), "amp": d["amp"].float(), "y": d["y"].long(),
            "onehot": F.one_hot(d["y"].long(), n_classes).float(), "n_classes": n_classes,
            "y_np": z["y"], "split": z["split"]}
    for name, s in (("train", TRAIN), ("val", VAL), ("test", TEST)):
        idx = np.flatnonzero(z["split"] == s)
        data[f"{name}_idx_np"] = idx
        data[f"{name}_idx"] = torch.from_numpy(idx).to(device)
    data["scale"] = float(d["mag"][data["train_idx"]].abs().max())
    return data


def subset(data, idx, x=None):
    """A split as the classifier wants it: x, ph, y (optionally with synthetic magnitudes x)."""
    return {"x": data["x"][idx] if x is None else x, "ph": data["ph"][idx], "y": data["y"][idx]}


class GanRun:
    """One GAN training run with checkpoints, per-checkpoint evaluation, resume and final test scores."""

    def __init__(self, cfg, data, feat_model, out_dir, deadline, device="cuda"):
        self.cfg, self.data, self.feat, self.dev = cfg, data, feat_model, device
        self.out, self.deadline = out_dir, deadline
        (out_dir / "ckpt").mkdir(parents=True, exist_ok=True)
        self.log = Logger(out_dir / "log.txt")
        self.metrics_path = out_dir / "metrics.json"
        self.metrics = load_json(self.metrics_path, [])
        torch.manual_seed(cfg["seed"])
        N, L = data["n_classes"], data["x"].shape[1]
        self.G = Generator(cfg["latent"], L, N, data["scale"]).to(device)
        self.D = Discriminator(L, N).to(device)
        self.G_ema = copy.deepcopy(self.G).eval()
        self.optG = torch.optim.Adam(self.G.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
        self.optD = torch.optim.Adam(self.D.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
        self.amp = amp_on(cfg.get("amp", False), device)
        self.scalerG = torch.amp.GradScaler(enabled=self.amp)
        self.scalerD = torch.amp.GradScaler(enabled=self.amp)
        self.skipped = 0.0                 # D steps with a non-finite loss (AMP skips those updates)
        ytr = data["y_np"][data["train_idx_np"]]
        self.sampler = ClassResampler(ytr, cfg["gamma"], seed=cfg["seed"], n_classes=N)
        self.it = 0
        self._real_cache = None

    # ---------------- state ----------------
    def save_state(self):
        torch_save({"G": self.G.state_dict(), "D": self.D.state_dict(), "G_ema": self.G_ema.state_dict(),
                    "optG": self.optG.state_dict(), "optD": self.optD.state_dict(), "sampler": self.sampler,
                    "scalerG": self.scalerG.state_dict(), "scalerD": self.scalerD.state_dict(),
                    "it": self.it, "rng_cpu": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if self.dev == "cuda" else None}, self.out / "last.pt")

    def load_state(self):
        p = self.out / "last.pt"
        if not p.exists():
            return False
        s = torch.load(p, map_location=self.dev, weights_only=False)
        self.G.load_state_dict(s["G"]); self.D.load_state_dict(s["D"]); self.G_ema.load_state_dict(s["G_ema"])
        self.optG.load_state_dict(s["optG"]); self.optD.load_state_dict(s["optD"])
        self.scalerG.load_state_dict(s["scalerG"]); self.scalerD.load_state_dict(s["scalerD"])
        self.sampler, self.it = s["sampler"], s["it"]
        restore_rng(s, self.dev)
        return True

    # ---------------- training ----------------
    def batch(self, idx_np):
        d = self.data
        j = d["train_idx"][torch.from_numpy(idx_np).to(self.dev)]
        return d["x"][j], (d["ph"][j], d["amp"][j], d["onehot"][j]), d["y"][j]

    def step(self):
        c, B = self.cfg, self.cfg["batch"]
        for _ in range(c["n_disc"]):
            x_r, cond, y = self.batch(self.sampler.sample(B))
            with torch.no_grad(), autocast(self.dev, self.amp):
                x_g = self.G(torch.randn(B, c["latent"], device=self.dev), *cond).float()
            with autocast(self.dev, self.amp):
                d_r, logit_r = self.D(x_r, *cond)
                d_g, logit_g = self.D(x_g, *cond)
            d_r, d_g = d_r.float(), d_g.float()
            gp = gradient_penalty(self.D, x_r, x_g, cond, self.scalerD, self.amp)
            h = F.cross_entropy(logit_r.float(), y) + F.cross_entropy(logit_g.float(), y)
            loss_d = d_g.mean() - d_r.mean() + c["lambda_gp"] * gp + c["xi"] * h + c["eps_drift"] * (d_r ** 2).mean()
            self.optD.zero_grad(set_to_none=True)
            self.scalerD.scale(loss_d).backward()
            self.scalerD.step(self.optD)
            self.scalerD.update()
            self.skipped = self.skipped + (~torch.isfinite(loss_d.detach())).float()
        _, cond, y = self.batch(self.sampler.sample(B))
        with autocast(self.dev, self.amp):
            x_g = self.G(torch.randn(B, c["latent"], device=self.dev), *cond)
            d_g, logit_g = self.D(x_g, *cond)
        d_g = d_g.float()
        loss_g = -d_g.mean() + c["xi"] * F.cross_entropy(logit_g.float(), y)
        self.optG.zero_grad(set_to_none=True)
        self.scalerG.scale(loss_g).backward()
        self.scalerG.step(self.optG)
        self.scalerG.update()
        with torch.no_grad():
            for pe, p in zip(self.G_ema.parameters(), self.G.parameters()):
                pe.lerp_(p, 1 - c["ema"])
        self.it += 1
        return torch.nan_to_num(torch.stack([loss_d.detach(), loss_g.detach(), (d_r.mean() - d_g.mean()).detach()]))

    # ---------------- evaluation of one checkpoint ----------------
    def real_cache(self):
        """Features of the real training set, per class, with their k-NN radii (computed once)."""
        if self._real_cache is None:
            d = self.data
            tr = d["train_idx"]
            f = extract_features(self.feat, d["x"][tr], d["ph"][tr])
            ytr = d["y"][tr]
            per = []
            for c in range(d["n_classes"]):
                fc = f[ytr == c]
                per.append((fc, knn_radii(fc, self.cfg["k"])))
            self._real_cache = (gauss_stats(f.cpu().numpy().astype(np.float64)), per)
        return self._real_cache

    def eval_checkpoint(self, G, it):
        c, d = self.cfg, self.data
        t0 = time.time()
        real_stats, per = self.real_cache()
        n_gen = len(d["train_idx"])
        # density & recall per class, repeated on fresh generated sets (mean over repetitions)
        reps = []
        for r in range(c["dr_reps"]):
            idx = balanced_idx(d, n_gen, seed=10_000 + r)
            fg = extract_features(self.feat, generate(G, d, idx, seed=20_000 + r), d["ph"][idx])
            yg = d["y"][idx]
            reps.append([prdc(per[k][0], fg[yg == k], c["k"], real_radii=per[k][1]) for k in range(d["n_classes"])])
        res = {"it": it}
        for m in ("density", "recall", "precision", "coverage"):
            arr = np.array([[cls[m] for cls in rep] for rep in reps])       # (reps, classes)
            res[m] = arr.mean(0).tolist()
            res[m + "_std"] = arr.std(0).tolist()
        # FID: 10 batches of 5000 generated curves (conditions at the natural class mix) vs the real train set
        fids = []
        for b in range(c["fid_batches"]):
            idx = balanced_idx(d, c["fid_batch_size"], seed=30_000 + b, gamma=0.0)
            fg = extract_features(self.feat, generate(G, d, idx, seed=40_000 + b), d["ph"][idx])
            fids.append(fid_from_stats(*real_stats, *gauss_stats(fg.cpu().numpy().astype(np.float64))))
        res["fid"] = float(np.mean(fids))
        # validation TSTR accuracy: classifiers trained on a balanced generated set, scored on real val
        idx = balanced_idx(d, n_gen, seed=50_000 + it)
        syn = subset(d, idx, x=generate(G, d, idx, seed=60_000 + it))
        r = fit_ensembles(syn, subset(d, d["val_idx"]), subset(d, d["test_idx"]), d["n_classes"],
                          runs=c["sel_runs"], ens=c["ens"], max_epochs=c["sel_max_epochs"],
                          patience=c["sel_patience"], seed=it, device=self.dev, log=lambda *a: None,
                          amp=c.get("amp", False))
        res["val_acc_runs"] = r["val_acc"]
        res["val_acc"] = float(np.mean(r["val_acc"]))
        res["test_acc_info"] = float(np.mean(r["test_acc"]))       # logged only, never used for selection
        res["eval_s"] = time.time() - t0
        return res

    def evaluate_pending(self):
        done = {m["it"] for m in self.metrics}
        for p in sorted((self.out / "ckpt").glob("G_ema_*.pt")):
            it = int(p.stem.split("_")[-1])
            if it in done:
                continue
            G = copy.deepcopy(self.G_ema)
            G.load_state_dict(torch.load(p, map_location=self.dev))
            m = self.eval_checkpoint(G.eval(), it)
            self.metrics.append(m)
            self.metrics.sort(key=lambda a: a["it"])
            save_json(self.metrics, self.metrics_path)
            self.log(f"checkpoint {it:,}: FID {m['fid']:.2f} | macro D {np.mean(m['density']):.3f} "
                     f"R {np.mean(m['recall']):.3f} | val TSTR acc {m['val_acc']:.3f} ({m['eval_s']:.0f} s)")

    # ---------------- the whole run ----------------
    def run(self, reserve_s):
        c = self.cfg
        if (self.out / "final.json").exists():
            self.log("already finished")
            return "done"
        if self.load_state():
            self.log(f"resumed at iteration {self.it:,}")
        self.evaluate_pending()
        t_save, t0, it0 = time.time(), time.time(), self.it
        hist, n_hist = 0.0, 0         # running sums stay on the GPU (no sync every step)
        while self.it < c["iters"]:
            if self.deadline.passed(reserve_s):
                self.save_state()
                self.log(f"time budget reached at iteration {self.it:,}: state saved, resume next session")
                return "paused"
            hist = hist + self.step()
            n_hist += 1
            if self.it % c["ckpt_every"] == 0:
                torch_save(self.G_ema.state_dict(), self.out / "ckpt" / f"G_ema_{self.it:07d}.pt")
                self.save_state()
                t_save = time.time()
                h = (hist / max(n_hist, 1)).tolist()
                speed = (self.it - it0) / (time.time() - t0)
                self.log(f"iter {self.it:,}: loss D {h[0]:.3f} G {h[1]:.3f} | W-dist est. {h[2]:.3f} | "
                         f"{speed:.1f} it/s" + (f" | amp scale {self.scalerD.get_scale():.0f}, skipped D steps "
                                                f"{float(self.skipped):.0f}" if self.amp else ""))
                with open(self.out / "train_curve.csv", "a") as f:
                    f.write(f"{self.it},{h[0]:.5f},{h[1]:.5f},{h[2]:.5f}\n")
                hist, n_hist = 0.0, 0
                self.evaluate_pending()
                t0, it0 = time.time(), self.it
            elif time.time() - t_save > c["save_every_s"]:
                self.save_state()
                t_save = time.time()
        self.save_state()
        self.evaluate_pending()
        return self.finish(reserve_s)

    def finish(self, reserve_s):
        """Model selection (val TSTR accuracy and G-score) and the final TSTR test scores (15 runs x 5 CNNs)."""
        c, d = self.cfg, self.data
        its = [m["it"] for m in self.metrics]
        G_sc = g_score([m["density"] for m in self.metrics], [m["recall"] for m in self.metrics])
        for m, g in zip(self.metrics, G_sc):
            m["g_score"] = float(g)
        save_json(self.metrics, self.metrics_path)
        choice = {"val_acc": its[int(np.argmax([m["val_acc"] for m in self.metrics]))],
                  "g_score": its[int(np.argmax(G_sc))]}
        final = load_json(self.out / "final_partial.json", {})
        for crit, it in choice.items():
            if crit in final:
                continue
            if self.deadline.passed(reserve_s / 2):
                self.log("time budget reached before the final evaluation; it resumes next session")
                return "paused"
            G = copy.deepcopy(self.G_ema)
            G.load_state_dict(torch.load(self.out / "ckpt" / f"G_ema_{it:07d}.pt", map_location=self.dev))
            idx = balanced_idx(d, len(d["train_idx"]), seed=70_000)
            syn = subset(d, idx, x=generate(G.eval(), d, idx, seed=80_000))
            r = fit_ensembles(syn, subset(d, d["val_idx"]), subset(d, d["test_idx"]), d["n_classes"],
                              runs=c["final_runs"], ens=c["ens"], max_epochs=c["final_max_epochs"],
                              patience=c["final_patience"], seed=1, device=self.dev, log=self.log, tag=crit,
                              amp=c.get("amp", False))
            r["checkpoint"] = it
            final[crit] = r
            save_json(final, self.out / "final_partial.json")
            self.log(f"{crit}: checkpoint {it:,} -> TSTR test accuracy "
                     f"{100 * np.mean(r['test_acc']):.2f} +- {100 * np.std(r['test_acc']):.2f}")
            # a few generated curves per class from this checkpoint, for the figures
            show = balanced_idx(d, 16 * d["n_classes"], seed=90_000)
            np.savez(self.out / f"samples_{crit}.npz", idx=show.cpu().numpy(),
                     x=generate(G, d, show, seed=91_000).cpu().numpy())
        save_json(final, self.out / "final.json")
        self.log("finished")
        return "done"


In [ ]:
%%writefile astra/run_gan.py
"""Worker for the GAN notebook: runs a queue of jobs on one GPU, skipping finished ones.

python -m astra.run_gan --queue trtr,gan_u_s0 --deadline <unix time> [--smoke]
"""
import argparse
import copy
import os
import time
import traceback
from pathlib import Path

import numpy as np
import torch

from .classifier import CNNEnsemble, fit_ensembles
from .common import WORK, Deadline, Logger, load_json, save_json
from .gan import GanRun, load_data, subset
from .warping import make_warp

# Settings from the paper (Sec. 4.3-4.4); the ones it does not state are marked "ours".
GAN = dict(latent=16, iters=400_000, n_disc=5, lr=1e-4, xi=1e-3, eps_drift=1.0, ema=0.999, ckpt_every=10_000,
           batch=64,            # ours: batch size is not given in the paper (64 as in WGAN-GP)
           amp=True,            # ours: fp16 mixed precision on the GPU (fp32 needs ~24 h per run on a T4)
           lambda_gp=10.0,      # ours: GP weight not given (10 as in WGAN-GP)
           k=5, dr_reps=5,      # ours: k for the k-NN balls (Naeem et al. default); 5 repetitions as in Fig. 7
           fid_batches=10, fid_batch_size=5000,
           sel_runs=3, ens=5, sel_max_epochs=6, sel_patience=2,        # val TSTR per checkpoint (ours: 3 runs)
           final_runs=15, final_max_epochs=10, final_patience=3,       # Table 5: 15 runs of 5-CNN ensembles
           # (ours: epochs and patience; measured: the classifier's validation loss bottoms out after ~2 epochs)
           save_every_s=1800)
JOBS = {
    "gan_gamma025_s0": dict(gamma=0.25, seed=0),     # gamma-GAN (rows G, H of Table 5)
    "gan_u_s0": dict(gamma=1.0, seed=0),             # u-GAN (rows E, F)
    "gan_gamma025_s1": dict(gamma=0.25, seed=1),     # extra seeds of the gamma-GAN
    "gan_gamma025_s2": dict(gamma=0.25, seed=2),
    "gan_gamma0_s0": dict(gamma=0.0, seed=0),        # no resampling (the gamma = 0 curve of Fig. 4)
}
TRTR_ROWS = {"A_oversampling": None, "B_soft_time_warping": "time", "C_soft_mag_warping": "mag",
             "D_soft_mixed_warping": "mixed"}
SMOKE = dict(iters=300, ckpt_every=100, dr_reps=1, fid_batches=2, fid_batch_size=500, sel_runs=1,
             sel_max_epochs=2, sel_patience=1, final_runs=2, final_max_epochs=2, final_patience=1, save_every_s=60)


def run_trtr(data, runs_dir, deadline, smoke, log, dev):
    """Rows A-D of Table 5: classifiers trained on real data (balanced by oversampling, +- soft warping)."""
    out = runs_dir / "trtr" / "trtr.json"
    res = load_json(out, {})
    kw = dict(runs=2, ens=5, max_epochs=2, patience=1) if smoke else \
        dict(runs=GAN["final_runs"], ens=GAN["ens"], max_epochs=GAN["final_max_epochs"], patience=GAN["final_patience"])
    tr, va, te = (subset(data, data[f"{s}_idx"]) for s in ("train", "val", "test"))
    for row, mode in TRTR_ROWS.items():
        if row in res:
            continue
        if deadline.passed(1800):
            log("time budget reached; the remaining TRTR rows run next session")
            return "paused"
        t0 = time.time()
        r = fit_ensembles(tr, va, te, data["n_classes"], warp=make_warp(mode), seed=0, device=dev, log=log,
                          tag=row, **kw)
        r["minutes"] = (time.time() - t0) / 60
        res[row] = r
        save_json(res, out)
        log(f"{row}: test accuracy {100 * np.mean(r['test_acc']):.2f} +- {100 * np.std(r['test_acc']):.2f} "
            f"({r['minutes']:.1f} min)")
    return "done"


def benchmark(data, feat, dev, tmp_dir, latent=16):
    """SMOKE only: times the real-size GAN step and classifier epochs, to estimate every job's hours."""
    out = {}
    for amp in (False, True):           # full precision first, then mixed precision (the one the jobs use)
        run = GanRun(dict(GAN, gamma=0.25, seed=0, amp=amp, latent=latent), data, feat, tmp_dir,
                     Deadline(time.time() + 1e6), dev)
        for _ in range(20):
            run.step()
        if dev == "cuda":
            torch.cuda.synchronize()
        t = time.time()
        for _ in range(60):
            last = run.step()
        if dev == "cuda":
            torch.cuda.synchronize()
        out["gan_it_per_s" if amp else "gan_it_per_s_fp32"] = 60 / (time.time() - t)
    out["amp_skipped_D_steps"] = float(run.skipped)
    out["amp_scale_D"] = run.scalerD.get_scale() if run.amp else 0.0
    out["amp_last_losses_D_G_W"] = [round(v, 3) for v in last.tolist()]
    t = time.time()
    run.cfg = dict(run.cfg, sel_max_epochs=1)
    run.eval_checkpoint(copy.deepcopy(run.G_ema), 0)
    out["ckpt_eval_1epoch_s"] = time.time() - t
    tr, va, te = (subset(data, data[f"{s}_idx"]) for s in ("train", "val", "test"))
    for runs in (GAN["sel_runs"], GAN["final_runs"]):
        t = time.time()
        r = fit_ensembles(tr, va, te, data["n_classes"], runs=runs, ens=GAN["ens"], max_epochs=1, device=dev,
                          log=lambda *a: None)
        out[f"clf_epoch_s_{runs * GAN['ens']}cnn"] = time.time() - t
    out["clf_1epoch_test_acc"] = float(np.mean(r["test_acc"]))
    t = time.time()
    fit_ensembles(tr, va, te, data["n_classes"], runs=GAN["sel_runs"], ens=GAN["ens"], max_epochs=1, device=dev,
                  log=lambda *a: None, amp=False)
    out[f"clf_epoch_s_{GAN['sel_runs'] * GAN['ens']}cnn_fp32"] = time.time() - t
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--queue", required=True)
    ap.add_argument("--deadline", type=float, required=True)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--runs-dir", default=str(WORK / "runs"))
    ap.add_argument("--data", default=str(WORK / "data" / "catalina_L100.npz"))
    ap.add_argument("--latent", type=int, default=GAN["latent"])       # 16 for Catalina, 8 for ZTF (Sec. 4.2)
    a = ap.parse_args()
    runs_dir = Path(a.runs_dir)
    tiny = a.smoke or os.environ.get("ASTRA_LOCAL_TEST") == "1"     # my offline CPU test runs tiny jobs
    deadline = Deadline(a.deadline)
    dev = "cuda" if torch.cuda.is_available() else "cpu"      # cpu only in my offline tests
    torch.backends.cudnn.benchmark = True
    data = load_data(a.data, dev)
    feat = CNNEnsemble(1, data["n_classes"], data["x"].shape[1]).to(dev)
    feat.load_state_dict(torch.load(runs_dir / "feature_cnn.pt", map_location=dev))
    feat.eval()
    for job in a.queue.split(","):
        status_path = runs_dir / job / "status.json"
        if (load_json(status_path, {}) or {}).get("state") == "done":
            print(f"[{job}] already done, skipping", flush=True)
            continue
        log = Logger(runs_dir / job / "log.txt")
        log(f"start {job} ({'tiny' if tiny else 'real run'})")
        t0 = time.time()
        if dev == "cuda":
            torch.cuda.reset_peak_memory_stats()
        try:
            if job == "trtr":
                state = run_trtr(data, runs_dir, deadline, tiny, log, dev)
            else:
                cfg = dict(GAN, **JOBS[job], **(SMOKE if tiny else {}), latent=a.latent)
                save_json(cfg, runs_dir / job / "config.json")
                state = GanRun(cfg, data, feat, runs_dir / job, deadline, dev).run(reserve_s=600)
        except Exception:
            log(traceback.format_exc())
            state = "failed"
        prev = load_json(status_path, {}) or {}
        save_json({"state": state, "hours_this_session": (time.time() - t0) / 3600,
                   "hours_total": prev.get("hours_total", 0) + (time.time() - t0) / 3600,
                   "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else 0}, status_path)
        log(f"{job}: {state}")
        if state == "paused":
            break                 # out of time: the rest of the queue waits for the next session


if __name__ == "__main__":
    main()


## 1 · Data (Sec. 3 of the paper)

### Catalina Surveys DR1 periodic variables

The catalog (47,055 periodic variables with period, amplitude and type) and all their photometry (MJD, magnitude, error) are public on the Catalina Surveys site. The paper keeps 6 classes (its Table 1) and pre-processes every light curve in four steps:
1. **Period folding**: phase = (t mod T) / T, so the curve's shape becomes visible in [0, 1).
2. **Outlier filtering**: a sliding window of 20 points (in phase order) drops points with z-score > 3, done twice. Then a curve is dropped if more than 90% of its magnitudes lie outside the class median ± k·σ. The paper does not give k, and **k = 2 reproduces its class counts** (table below).
3. **Time sub-sampling**: keep L = 100 randomly chosen observations. This gives a fixed length, so convolutions can be used.
4. **Median centring**: subtract the curve's median magnitude, since the generator's tanh output is symmetric around 0.

### ZTF (only when `DATASET = "ztf"`)

The paper uses the periodic variable stars of ALeRCE's ZTF training set, with 40 points per curve. Neither their processed data nor their code is public, and the paper does not state the band, the dates, or where the period comes from. We recovered these from the paper itself: the real light curves it plots (Figs. 8 and 9, and the same in the first author's thesis) were digitised and matched against ALeRCE's detections, which identified all 21 stars shown. What they tell us:
- **One curve per star and band.** Each curve is single-band, and the dataset holds both g curves and r curves (14 and 7 of the 21).
- **Period:** the `period` column of ALeRCE's internal label file, i.e. the period given by the catalogue the label came from. It equals the folding period of all 21 curves to five digits. For eclipsing binaries it is the full orbital period (ALeRCE's own `Multiband_period` is half of it). Most Gaia-labelled stars have no period in that file (78% of the RR Lyrae, 74% of the LPVs); they cannot be folded and drop out, which is why the paper has so few of both.
- **Dates:** no plotted point is later than 19 July 2020 (MJD 59048.3), and many are that late: the data is one snapshot of that date.
- **Filtering:** only the magnitude threshold of 30.

So the dataset is rebuilt from public sources as follows:
- **Labels:** ALeRCE's labeled set (Sánchez-Sáez et al. 2021, Zenodo record 4279623): 85,809 periodic stars in the classes EB, RRL, LPV, DSCT and CEP.
- **Periods:** `ZTF_labels_v8.0.1.csv`, a public copy of ALeRCE's label file (Pérez-Carrasco et al. 2023, AnomalyALeRCE repository).
- **Light curves:** ALeRCE's public TAP service (table `ztf.detection`), corrected magnitudes below 30, up to MJD 59048.4, repeated epochs dropped. One curve per star and band with at least 40 points.
- Then, as for Catalina: fold, keep 40 random points, median-centre. No z-score or class-range filter, since the paper applies only the magnitude threshold to ZTF.

This rule gives EB 31,174, RRL 18,748 and LPV 5,470 curves, against the paper's 31,477, 18,729 and 5,245. For DSCT and CEP it gives 768 and 716 against the paper's 507 and 471, which we could not explain. We therefore **draw the paper's number of curves per class at random** where more qualify (for EB, all 31,174 are used). The class imbalance and the split sizes then equal the paper's; most stars should be the paper's too, but the 40 points of each curve are a new random draw.

Checked offline on the classifier trained on real data (row A), over random draws of the curves: this recipe gives the paper's accuracy (72.61) within the spread between draws, about 2 points.

The download takes over an hour, so it is done once offline (`build/ztf/download_ztf.py`, `add_label_periods.py`, then `make_ztf_dataset.py`) and the resulting `ztf_L40.npz` (24 MB) is attached as a Kaggle dataset.

### Split (Sec. 4.1)

Validation and test each hold 20% of the smallest class, *for every class*: 36 stars per class for Catalina (CEP has 181) and 94 curves per class for ZTF (CEP has 471). For ZTF the split is over curves, as in the paper, so the g and the r curve of one star can fall on different sides. Both sets are balanced, so accuracy equals balanced accuracy. Everything else is training data, still heavily imbalanced (Catalina: EBSD/D ≈ 30k vs CEP ≈ 109).

In [ ]:
# ---- Get the pre-processed light curves (made once; later sessions reuse the saved arrays) ----
import matplotlib.pyplot as plt
import pandas as pd
from astra import catalina, ztf
from astra.common import find_input

NPZ = WORK / "data" / f"{DATASET}_L{L}.npz"
if not NPZ.exists() and DATASET == "catalina":           # download + pre-process, about 3 minutes
    out, _ = catalina.preprocess(WORK / "data", L=L, class_range_k=CLASS_RANGE_K, seed=DATA_SEED)
    np.savez_compressed(NPZ, **out)
    (WORK / "data" / "AllVar.phot.gz").unlink(missing_ok=True)      # raw photometry no longer needed
if not NPZ.exists() and DATASET == "ztf":                # attached: the ready file, or the raw download
    ready, raw = find_input(NPZ.name), find_input("ztf_periodic_raw.npz")
    assert ready or raw, "ZTF: use Add Input to attach the dataset with ztf_L40.npz (see the data section above)"
    if ready:
        shutil.copy2(ready, NPZ)
    else:
        out, _ = ztf.preprocess(raw, L=L, seed=DATA_SEED)
        np.savez_compressed(NPZ, **out)
Z = np.load(NPZ)
CLASSES = [str(c) for c in Z["classes"]]
paper = (catalina if DATASET == "catalina" else ztf).PAPER_COUNTS
table = pd.DataFrame({"class": CLASSES, "ours": np.bincount(Z["y"]), "paper": [paper[c] for c in CLASSES]})
table["ours/paper"] = (table.ours / table.paper).round(3)
for name, s in (("train", 0), ("val", 1), ("test", 2)):
    table[name] = np.bincount(Z["y"][Z["split"] == s], minlength=len(CLASSES))
table.to_csv(TABLES / "table2_class_counts.csv", index=False)
print(table.to_string(index=False))
print(f"\n{len(Z['y']):,} light curves of {L} points | train {np.sum(Z['split'] == 0):,}, "
      f"val {np.sum(Z['split'] == 1)}, test {np.sum(Z['split'] == 2)}")

# a few pre-processed curves per class (like Fig. 1b); magnitudes grow downwards, so the y axis is flipped
fig, ax = plt.subplots(2, len(CLASSES), figsize=(3 * len(CLASSES), 5), sharex=True)
rng = np.random.default_rng(0)
for c, name in enumerate(CLASSES):
    for r, i in enumerate(rng.choice(np.flatnonzero(Z["y"] == c), 2, replace=False)):
        ax[r, c].plot(Z["phase"][i], Z["mag"][i], ".", ms=3)
        ax[r, c].invert_yaxis()
        ax[r, c].set_title(f"{name} (P = {Z['period'][i]:.3g} d)", fontsize=9)
fig.supxlabel("phase"); fig.supylabel("centred magnitude")
fig.tight_layout(); fig.savefig(FIGS / "data_examples.png", dpi=110); plt.show()

## 2 · Self-tests (seconds)

Cheap checks that would otherwise only show up hours into a run:
- the resampler draws classes with the probabilities of Eq. 12;
- the grouped network gives exactly the same outputs as separate CNNs, and one member never sees another member's data;
- warping keeps phases in order and inside the window;
- the metrics behave as expected on a copy, an independent sample and a collapsed generator;
- G and D produce the right shapes, and the gradient penalty back-propagates.

In [ ]:
# ---- Self-tests ----
from astra import sampling, classifier, warping, metrics, gan

y = np.array([0] * 900 + [1] * 90 + [2] * 10)
for g in (0.0, 0.25, 1.0):
    rs = sampling.ClassResampler(y, g, seed=1)
    assert np.allclose(np.bincount(y[rs.sample(100_000)], minlength=3) / 100_000, rs.p, atol=0.01)

M, N = 3, len(CLASSES)
ens = classifier.CNNEnsemble(M, N, L, dropout=0.0).train()
x = torch.randn(M, 16, 2, L)
out = ens(x)
for m in range(M):                  # every member == the same CNN built from nn.Conv1d / nn.Linear
    assert torch.allclose(classifier.reference_cnn(ens, m).train()(x[m]), out[m], atol=1e-4)
xg = torch.randn(M, 8, 2, L, requires_grad=True)
ens(xg)[0].sum().backward()
assert xg.grad[1:].abs().max() == 0, "members are independent"

ph = torch.sort(torch.rand(3, 50, L), -1).values
xb = torch.stack([torch.randn(3, 50, L), ph], 2)
for mode in ("time", "mag", "mixed"):
    w = warping.soft_warp(xb, mode)
    assert (torch.diff(w[..., 1, :], dim=-1) >= -1e-6).all() and w[..., 1, :].max() <= 1

a = torch.randn(2000, 16)
assert abs(metrics.prdc(a, a.clone())["density"] - 1.2) < 1e-4                 # a copy: (k+1)/k
same = metrics.prdc(a, torch.randn(2000, 16))
assert abs(same["density"] - 1) < 0.15 and same["recall"] > 0.8
collapsed = metrics.prdc(a, 0.05 * torch.randn(2000, 16))
assert collapsed["recall"] < 0.2, "a collapsed generator has low recall"

data_cpu = gan.load_data(NPZ, "cpu")
G = gan.Generator(LATENT, L, N, data_cpu["scale"]); D = gan.Discriminator(L, N)
j = data_cpu["train_idx"][:8]
cond = (data_cpu["ph"][j], data_cpu["amp"][j], data_cpu["onehot"][j])
xg = G(torch.randn(8, LATENT), *cond)
assert xg.shape == (8, L)
gan.gradient_penalty(D, data_cpu["x"][j], xg.detach(), cond).backward()
# the resume path on this device (reported, not asserted: a resumed job continues either way)
from astra.common import restore_rng, torch_save
torch_save({"rng_cpu": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if DEV == "cuda" else None}, WORK / "rng_test.pt")
first = torch.rand(3, device=DEV)
ok = restore_rng(torch.load(WORK / "rng_test.pt", map_location=DEV, weights_only=False), DEV)
print("resume check: random-number state", "restored" if ok and torch.equal(first, torch.rand(3, device=DEV))
      else "NOT restored (a resumed GAN continues with fresh random numbers)")
(WORK / "rng_test.pt").unlink()
print(f"all self-tests passed | G {sum(p.numel() for p in G.parameters()) / 1e6:.2f}M params, "
      f"D {sum(p.numel() for p in D.parameters()) / 1e6:.2f}M | tanh scale s = {data_cpu['scale']:.2f} mag")
del data_cpu, G, D, ens

## 3 · The feature classifier (Sec. 4.3)

Density, recall, FID and the G-score all compare real and generated curves in a **feature space**: the output of the last convolution block (4 × 64 = 256 numbers) of one base CNN trained on the balanced real training set. It is trained once, here, and shared by all GAN jobs so their metrics are comparable. Its training and validation losses are plotted below to check for over- or under-fitting.

In [ ]:
# ---- Train (or reuse) the feature CNN ----
from astra import gan as gan_mod

FEAT = RUNS / "feature_cnn.pt"
data = gan_mod.load_data(NPZ, DEV)
tr, va, te = (gan_mod.subset(data, data[f"{s}_idx"]) for s in ("train", "val", "test"))
if not FEAT.exists():
    torch.manual_seed(0)
    feat, curve = classifier.train_feature_cnn(tr, va, len(CLASSES), max_epochs=3 if SMOKE else 20, device=DEV)
    torch.save(feat.state_dict(), FEAT)
    json.dump(curve, open(RUNS / "feature_cnn_curve.json", "w"))
feat = classifier.CNNEnsemble(1, len(CLASSES), L).to(DEV)
feat.load_state_dict(torch.load(FEAT, map_location=DEV)); feat.eval()
curve = np.array(json.load(open(RUNS / "feature_cnn_curve.json")))
p = classifier.predict_proba(feat, classifier.stack_input(te["x"], te["ph"]), amp=False)[0].argmax(1).cpu().numpy()
print(f"feature CNN: {len(curve)} epochs | test accuracy of this single CNN: {np.mean(p == te['y'].cpu().numpy()):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(curve[:, 0], label="train (balanced)"); ax[0].plot(curve[:, 1], label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy"); ax[0].legend()
ax[1].plot(curve[:, 2]); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation accuracy")
fig.tight_layout(); fig.savefig(FIGS / "feature_cnn_curves.png", dpi=110); plt.show()
del data

In [ ]:
# ---- SMOKE only: time the real-size jobs and estimate each GPU queue's hours ----
if SMOKE:
    from astra import run_gan
    data = gan_mod.load_data(NPZ, DEV)
    b = run_gan.benchmark(data, feat, DEV, WORK / "bench_tmp", LATENT)
    shutil.rmtree(WORK / "bench_tmp", ignore_errors=True)
    del data
    c = run_gan.GAN
    n_ckpt = c["iters"] // c["ckpt_every"]
    ep_sel, ep_fin = b[f"clf_epoch_s_{c['sel_runs'] * c['ens']}cnn"], b[f"clf_epoch_s_{c['final_runs'] * c['ens']}cnn"]
    ep_sel32 = b[f"clf_epoch_s_{c['sel_runs'] * c['ens']}cnn_fp32"]
    SEL_EPOCHS, FIN_EPOCHS = 4, 5            # typical epochs before early stopping ends a training
    est = {"gan training": c["iters"] / b["gan_it_per_s"] / 3600,
           "checkpoint evaluations": n_ckpt * (b["ckpt_eval_1epoch_s"] + (SEL_EPOCHS - 1) * ep_sel) / 3600,
           "final TSTR (2 criteria)": 2 * FIN_EPOCHS * ep_fin / 3600}
    gan_h = sum(est.values())
    trtr_h = 4 * FIN_EPOCHS * ep_fin * 1.1 / 3600
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in b.items()})
    print(f"\nmixed precision: GAN {b['gan_it_per_s']:.1f} it/s (full precision {b['gan_it_per_s_fp32']:.1f}), "
          f"{b['amp_skipped_D_steps']:.0f} skipped D steps in 80; classifier epoch (15 CNNs) {ep_sel:.1f} s "
          f"(full precision {ep_sel32:.1f} s)")
    print("one GAN job ≈ " + ", ".join(f"{k} {v:.1f} h" for k, v in est.items()) + f" -> {gan_h:.1f} h")
    print(f"trtr (rows A-D) ≈ {trtr_h:.1f} h")
    for gpu, q in QUEUES.items():
        tot = sum(trtr_h if j == "trtr" else gan_h for j in q)
        print(f"GPU {gpu}: {q} ≈ {tot:.1f} h  (≈ {tot / (SESSION_HOURS - 0.7):.1f} sessions)")
    json.dump({"bench": b, "gan_job_h": gan_h, "trtr_h": trtr_h}, open(TABLES / "benchmark.json", "w"), indent=1)

## 4 · Training: one worker per GPU

Each GPU runs `python -m astra.run_gan` on its queue:
- `trtr` trains the classifiers of rows A–D (15 runs × 5 CNNs each, on real data). The 75 CNNs of a row train side by side as one network with a member axis.
- A `gan_*` job trains one GAN for 400k generator iterations, saving the EMA generator every 10k iterations. For every checkpoint it measures FID, per-class density and recall (5 repetitions), and the validation TSTR accuracy (3 runs × 5 CNNs trained on 43k generated curves).
- At the end, the G-score is computed over all checkpoints. The best checkpoint by each criterion (validation accuracy, G-score) gets the final TSTR test score: 15 runs × 5 CNNs.

In the real run, a 2-minute *canary* first runs a tiny version of each GPU's first job, so a broken setup fails early instead of after hours. The cell prints a short status line every 10 minutes; full logs are in `runs/<job>/log.txt`.

In [ ]:
# ---- Launch the two GPU workers and wait ----
RESULTS_RESERVE_H = 0.5          # kept free for the results cells and Kaggle's output saving
deadline = RUN_STARTED + SESSION_HOURS * 3600 - RESULTS_RESERVE_H * 3600


def launch(queues, runs_dir, smoke):
    procs = {}
    for gpu, q in queues.items():
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONPATH=str(WORK))
        f = open(runs_dir / f"worker_gpu{gpu}.log", "a")
        cmd = [sys.executable, "-m", "astra.run_gan", "--queue", ",".join(q), "--deadline", str(deadline),
               "--runs-dir", str(runs_dir), "--data", str(NPZ), "--latent", str(LATENT)] + (["--smoke"] if smoke else [])
        procs[gpu] = (subprocess.Popen(cmd, env=env, stdout=f, stderr=subprocess.STDOUT, cwd=WORK), q)
    return procs


def last_line(job, runs_dir):
    p = runs_dir / job / "log.txt"
    lines = p.read_text(encoding="utf-8").strip().splitlines() if p.exists() else []
    return lines[-1][:150] if lines else "(not started)"


def wait(procs, runs_dir, every_s):
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(every_s)
        h = (time.time() - RUN_STARTED) / 3600
        for gpu, (p, q) in procs.items():
            cur = next((j for j in q if (json.loads((runs_dir / j / "status.json").read_text())["state"]
                        if (runs_dir / j / "status.json").exists() else "") not in ("done", "failed")), q[-1])
            print(f"[{h:5.2f} h] GPU {gpu} {'running' if p.poll() is None else 'exited '} | {cur}: "
                  f"{last_line(cur, runs_dir)}", flush=True)
    return {gpu: p.returncode for gpu, (p, _) in procs.items()}


if not SMOKE:          # canary: a tiny version of each GPU's first job, in a throw-away folder
    canary = WORK / "canary"
    canary.mkdir(exist_ok=True)
    shutil.copy2(RUNS / "feature_cnn.pt", canary / "feature_cnn.pt")
    codes = wait(launch({g: [next(j for j in q if j != "trtr")] for g, q in QUEUES.items()}, canary, True), canary, 20)
    states = {j.name: json.loads((j / "status.json").read_text())["state"] for j in canary.iterdir()
              if (j / "status.json").exists()}
    assert all(c == 0 for c in codes.values()) and all(s == "done" for s in states.values()), \
        f"canary failed: {codes} {states} (see {canary})"
    shutil.rmtree(canary)
    print("canary passed\n")

codes = wait(launch(QUEUES, RUNS, SMOKE), RUNS, 30 if SMOKE else 600)
print("workers finished with exit codes", codes)
for p in sorted(RUNS.glob("*/status.json")):
    print(f"  {p.parent.name:<18} {json.loads(p.read_text())}")

## 5 · Results: Table 5 (ours vs paper) and the paper's figures

The cell below reads whatever has finished so far (so it also works on a paused session) and produces:
- **Table 5**: test accuracy, mean ± std over 15 runs, next to the paper's numbers, with two-sided Welch's t-test p-values as in the paper. Macro-F1 and per-class recall are added for the rarity analysis.
- **Fig. 3/4-style** curves: validation TSTR accuracy and FID over GAN training, and the γ comparison.
- **Fig. 5/7/11-style** plots: per-class density and recall, the G-score curve, and the density-recall plane with the two model-selection choices marked.
- **Fig. 9-style** samples: real curves and generated curves with the *same* condition (phases, amplitude, class).
- The TRTR classifiers' loss curves, to check over- and under-fitting.

In [ ]:
# ---- Table 5 and figures from everything finished so far ----
from astra.metrics import welch_p

PAPER = {"catalina": {"A": 73.44, "B": 74.06, "C": 73.64, "D": 73.82, "E": 75.97, "F": 76.28, "G": 76.86, "H": 76.97},
         "ztf": {"A": 72.61, "B": 72.69, "C": 72.45, "D": 72.53, "E": 74.17, "F": 73.79, "G": 74.37, "H": 74.94}}[DATASET]
PAPER_SD = {"catalina": {"A": 1.22, "B": 1.04, "C": 1.79, "D": 1.50, "E": 0.94, "F": 0.74, "G": 1.09, "H": 0.79},
            "ztf": {"A": 0.69, "B": 0.99, "C": 0.70, "D": 0.69, "E": 0.62, "F": 0.50, "G": 0.51, "H": 0.44}}[DATASET]
load = lambda p: json.loads(p.read_text()) if p.exists() else None
trtr = load(RUNS / "trtr" / "trtr.json") or {}
rows = {}                                     # row -> (label, result dict)
for key, name in (("A", "A_oversampling"), ("B", "B_soft_time_warping"), ("C", "C_soft_mag_warping"),
                  ("D", "D_soft_mixed_warping")):
    if name in trtr:
        rows[key] = (f"TRTR {name[2:].replace('_', ' ')}", trtr[name])
gan_jobs = sorted(p.parent.name for p in RUNS.glob("gan_*/config.json"))
for job in gan_jobs:
    fin = load(RUNS / job / "final.json") or {}
    base = {"gan_u_s0": ("E", "F"), "gan_gamma025_s0": ("G", "H")}.get(job)
    for i, crit in enumerate(("val_acc", "g_score")):
        if crit in fin:
            key = base[i] if base else f"{job}:{crit}"
            rows[key] = (f"TSTR {job} [{crit}, ckpt {fin[crit]['checkpoint'] // 1000}k]", fin[crit])

rows = dict(sorted(rows.items()))            # A..H first, then the extra seeds
tab = []
for key, (label, r) in rows.items():
    acc = 100 * np.array(r["test_acc"])
    tab.append({"row": key, "method": label, "ours_acc": f"{acc.mean():.2f} ± {acc.std():.2f}",
                "paper_acc": f"{PAPER[key]:.2f} ± {PAPER_SD[key]:.2f}" if key in PAPER else "",
                "diff_vs_paper": round(acc.mean() - PAPER[key], 2) if key in PAPER else None,
                "p_vs_A": round(welch_p(acc, 100 * np.array(rows["A"][1]["test_acc"])), 4) if "A" in rows and key != "A" else None,
                "macro_F1": f"{100 * np.mean(r['test_f1']):.2f}", "runs": len(acc),
                "epochs(best, mean)": round(float(np.mean(r["best_epoch"])), 1)})
tab = pd.DataFrame(tab)
tab.to_csv(TABLES / "table5_ours_vs_paper.csv", index=False)
print(f"Table 5 ({DATASET}) — test accuracy %, mean ± std over runs")
print(tab.to_string(index=False) if len(tab) else "(no finished rows yet)")
if len(rows) > 1:
    keys = list(rows)
    pv = pd.DataFrame([[welch_p(rows[a][1]["test_acc"], rows[b][1]["test_acc"]) if a != b else np.nan for b in keys]
                       for a in keys], index=keys, columns=keys).round(4)
    pv.to_csv(TABLES / "welch_pvalues.csv")
    print("\nWelch's t-test p-values between rows:\n", pv.to_string())
    rec = pd.DataFrame({k: 100 * np.mean(r["test_recall_per_class"], 0) for k, (_, r) in rows.items()},
                       index=CLASSES).round(1)
    rec.to_csv(TABLES / "per_class_recall.csv")
    print("\nPer-class test recall % (classes from most to least common):\n", rec.to_string())

# ---- curves over GAN training (Fig. 3, 4, 7 style) ----
M = {j: load(RUNS / j / "metrics.json") for j in gan_jobs}
M = {j: m for j, m in M.items() if m}
if M:
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    for j, m in M.items():
        it = np.array([a["it"] for a in m]) / 1000
        va = np.array([a["val_acc"] for a in m]); vs = np.array([np.std(a["val_acc_runs"]) for a in m])
        ax[0].plot(it, 100 * va, label=j); ax[0].fill_between(it, 100 * (va - vs), 100 * (va + vs), alpha=.2)
        ax[1].plot(it, [a["fid"] for a in m], label=j)
        if "g_score" in m[0]:
            ax[2].plot(it, [a["g_score"] for a in m], label=j)
    for a, t in zip(ax, ("validation TSTR accuracy %", "FID (lower = closer)", "G-score (after training)")):
        a.set_xlabel("generator iterations (k)"); a.set_title(t); a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIGS / "gan_training_curves.png", dpi=110); plt.show()

    fig, ax = plt.subplots(1, len(M), figsize=(5 * len(M), 4), squeeze=False)
    for a, (j, m) in zip(ax[0], M.items()):
        D = np.array([x["density"] for x in m]).mean(1); R = np.array([x["recall"] for x in m]).mean(1)
        sc = a.scatter(R, D, c=[x["it"] / 1000 for x in m], cmap="viridis")
        fin = load(RUNS / j / "final.json") or {}
        for crit, mk in (("val_acc", "s"), ("g_score", "*")):
            if crit in fin:
                k = [x["it"] for x in m].index(fin[crit]["checkpoint"])
                a.scatter(R[k], D[k], marker=mk, s=250, facecolors="none", edgecolors="r", label=crit)
        a.set_xlabel("macro recall (diversity)"); a.set_ylabel("macro density (fidelity)"); a.set_title(j)
        a.legend(); plt.colorbar(sc, ax=a, label="iterations (k)")
    fig.tight_layout(); fig.savefig(FIGS / "density_recall_plane.png", dpi=110); plt.show()

    j0 = "gan_gamma025_s0" if "gan_gamma025_s0" in M else next(iter(M))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    it = np.array([a["it"] for a in M[j0]]) / 1000
    for c, name in enumerate(CLASSES):
        ax[0].plot(it, [a["density"][c] for a in M[j0]], label=name)
        ax[1].plot(it, [a["recall"][c] for a in M[j0]], label=name)
    ax[0].set_title(f"{j0}: density per class"); ax[1].set_title(f"{j0}: recall per class")
    for a in ax:
        a.set_xlabel("generator iterations (k)"); a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIGS / "per_class_density_recall.png", dpi=110); plt.show()

# ---- real vs generated with the same conditions (Fig. 9 style) ----
for j in gan_jobs:
    s = RUNS / j / "samples_g_score.npz"
    if not s.exists():
        continue
    S = np.load(s)
    fig, ax = plt.subplots(2, len(CLASSES), figsize=(18, 5), sharex=True)
    for c, name in enumerate(CLASSES):
        k = np.flatnonzero(Z["y"][S["idx"]] == c)[:2]
        for r, i in enumerate(k):
            ax[r, c].plot(Z["phase"][S["idx"][i]], Z["mag"][S["idx"][i]], ".", ms=3, label="real")
            ax[r, c].plot(Z["phase"][S["idx"][i]], S["x"][i], ".", ms=3, label="generated")
            ax[r, c].invert_yaxis(); ax[r, c].set_title(name, fontsize=9)
    ax[0, 0].legend(fontsize=7); fig.suptitle(f"{j}, G-score checkpoint: real vs generated, same condition")
    fig.tight_layout(); fig.savefig(FIGS / f"samples_{j}.png", dpi=110); plt.show()

# ---- over/under-fitting of the TRTR classifiers ----
if trtr:
    fig, ax = plt.subplots(1, len(trtr), figsize=(4.5 * len(trtr), 3.2), squeeze=False)
    for a, (name, r) in zip(ax[0], trtr.items()):
        a.plot(r["curves"]["epoch"], r["curves"]["train_loss"], label="train")
        a.plot(r["curves"]["epoch"], r["curves"]["val_loss"], label="val")
        a.set_title(name, fontsize=9); a.set_xlabel("epoch"); a.legend()
    fig.tight_layout(); fig.savefig(FIGS / "trtr_loss_curves.png", dpi=110); plt.show()

In [ ]:
# ---- Deliverables check: is everything saved that later notebooks and the report need? ----
need = [NPZ, TABLES / "table2_class_counts.csv", RUNS / "feature_cnn.pt", FIGS / "data_examples.png",
        TABLES / "table5_ours_vs_paper.csv"]
jobs = [j for q in QUEUES.values() for j in q]
summary = {"session": SESSION, "smoke": SMOKE, "hours": round((time.time() - RUN_STARTED) / 3600, 2), "jobs": {}}
for j in jobs:
    st = load(RUNS / j / "status.json") or {"state": "not started"}
    summary["jobs"][j] = st
    if st["state"] == "done":
        need += [RUNS / "trtr" / "trtr.json"] if j == "trtr" else \
                [RUNS / j / f for f in ("final.json", "metrics.json", "config.json", "log.txt", "samples_g_score.npz")]
missing = [str(p.relative_to(WORK)) for p in need if not p.exists()]
json.dump(summary, open(TABLES / f"session{SESSION}_summary.json", "w"), indent=1, default=str)
print(json.dumps(summary["jobs"], indent=1, default=str))
out_gb = sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file()) / 1e9
print(f"\noutput size {out_gb:.2f} GB (Kaggle keeps up to 20 GB)")
print("ALL DELIVERABLES PRESENT" if not missing else f"MISSING: {missing}")
left = [j for j, s in summary["jobs"].items() if s["state"] != "done"]
print("next session: attach this version's output, set SESSION =", SESSION + 1, "| unfinished:", left if left else "none")

## 6 · Observations

Written from **Catalina, sessions 1 to 3** (real runs, T4 ×2, 9.9, 9.9 and 9.6 h; 5, 10 and 12 October 2026): all six jobs are finished, and their tables, figures and executed notebooks were read. **ZTF** is written from sessions 1 to 3 (9.9, 9.9 and 7.6 h; 12 and 15 October and the γ = 0 session after that): all six jobs are finished; the tables and logs of all three were read, the figures of session 1.

**Replication verdict.**

| Row | Ours (Catalina) | Paper | Within the paper's ±? |
|---|---|---|---|
| A, oversampling | 73.58 ± 1.05 | 73.44 ± 1.22 | yes |
| B, soft time warping | 74.38 ± 1.14 | 74.06 ± 1.04 | yes |
| C, soft magnitude warping | 74.29 ± 0.81 | 73.64 ± 1.79 | yes |
| D, soft mixed warping | 74.51 ± 1.24 | 73.82 ± 1.50 | yes |
| E, u-GAN, chosen by validation accuracy (checkpoint 10k) | 76.48 ± 0.96 | 75.97 ± 0.94 | yes |
| F, u-GAN, chosen by G-score (checkpoint 10k) | 76.57 ± 1.13 | 76.28 ± 0.74 | yes |
| G, γ-GAN, chosen by validation accuracy (checkpoint 10k) | 76.33 ± 0.86 | 76.86 ± 1.09 | yes |
| H, γ-GAN, chosen by G-score (checkpoint 20k) | 74.60 ± 1.23 | 76.97 ± 0.79 | **no, 2.4 below** |

- Rows A–D reproduce the paper: every mean is within 0.7 points of the paper's and inside its ±.
- The warping rows are 0.7–0.9 points above row A, with Welch p-values of 0.06, 0.06 and 0.04 against A. The paper's own differences are of the same size (0.2–0.6). Neither is strong evidence that soft warping helps.
- Rows E, F and G reproduce the paper: each is within 0.6 points of the paper's value and inside its ±. Each is 2.8–3.0 points above row A with Welch p < 0.0001, so the paper's main claim, that a classifier trained on GAN samples beats one trained on the oversampled real set, holds here.
- Row H does not reproduce: 74.60 against 76.97. It is still above row A (+1.0, p = 0.026), but 1.7–2.0 below rows E–G (p ≤ 0.0002). The paper's best row is our worst GAN row.
- The paper's two orderings are not reproduced by this seed. (1) G-score against validation accuracy: for the u-GAN both criteria picked the same checkpoint, so rows E and F are one generator evaluated twice, and their difference of 0.09 (p = 0.82) is the repeat noise of the final evaluation, not a comparison of criteria. For the γ-GAN the G-score's choice is worse, not better. (2) γ-GAN against u-GAN: rows E, F and G do not differ (p = 0.52–0.66).
- All of this is one seed per GAN. Session 3 gave the two other seeds, and they are lower.
- γ-GAN chosen by validation accuracy, seeds 0 / 1 / 2: **76.33 / 74.32 / 75.25** (the paper: 76.86). Chosen by the G-score: **74.60 / 73.58 / 73.09** (the paper: 76.97). The spread between seeds, 1 to 2 points, is as large as the differences between the paper's four GAN rows, each of which comes from one training run.
- Against row A: seed 2 is clearly above (p = 0.0004), seed 1 is not (p = 0.07), and of the three G-score rows only seed 0's is (p = 0.03, 1.0, 0.31).
- γ = 0, no resampling (session 3): 72.8 with both criteria, 0.8 below row A and 1.5 to 3.5 below the γ = 0.25 rows chosen by validation accuracy. Its validation accuracy peaks later (82% at 40–50k) and then falls to 67–70% between 140k and 200k, far below any γ = 0.25 run, and its FID rises to 1.65. On Catalina the resampling is what keeps the GAN useful, which is the direction of the paper's Fig. 4.
- CEP recall is above row A's 24.3 in every GAN row of every seed (28.9 to 42.2). DSCT is above A's 63.5 only for γ = 0.25 chosen by validation accuracy (66.5 to 68.1).
- **ZTF, seed 0 (session 1).** Rows A–D: 73.21 / 73.72 / 73.26 / 73.30 against the paper's 72.61 / 72.69 / 72.45 / 72.53, so 0.6 to 1.0 above. Rows E–H: 73.33 / 73.43 / 72.48 / 70.85 against 74.17 / 73.79 / 74.37 / 74.94. **No GAN row beats row A on ZTF**: E and F equal it (p = 0.70 and 0.49), G is 0.7 below (p = 0.02) and H 2.4 below. In the paper all four are 1.2 to 2.3 above. The paper's ZTF result is not reproduced by this seed. Our ZTF set is rebuilt from public data and is not the paper's file; that can explain the offset of the reference rows, not the missing gain.
- **ZTF, seeds 1 and 2 (session 2).** γ-GAN chosen by validation accuracy: 71.49 (50k) and 72.64 (350k); chosen by the G-score: 70.50 and 71.65 (30k both). With seed 0's 72.48 and 70.85, **no γ-GAN row of any seed beats row A on ZTF**: the validation-accuracy rows are 0.6 to 1.7 below it (p = 0.02, 0.0001, 0.06), the G-score rows 1.6 to 2.7 below (p ≤ 0.0002). The paper reports 74.37 and 74.94. The missing ZTF gain is not one unlucky seed. The G-score picked 30k in all three seeds, and its row is again the lower one (by 1.0 to 1.6 points).
- **ZTF, γ = 0, no resampling (session 3).** 70.65 ± 0.97 chosen by validation accuracy (80k) and 71.04 ± 0.70 by the G-score (70k): 2.2 to 2.6 below row A (p < 0.0001); its validation-accuracy row is also below the γ = 0.25 validation-accuracy rows of all three seeds (p ≤ 0.045). As on Catalina, the resampling does better than none; on ZTF neither beats the oversampled real set. Its validation accuracy stays between 0.69 and 0.735 from 10k to 390k (highest at 80k), while FID keeps improving (0.66 at 10k → 0.18) and macro recall falls after 50k (0.76 → 0.55): FID again misses what the classifier sees. Rare-class recall: DSCT 64.1, CEP 40.2 against row A's 67.9 and 44.6.
- **ZTF verdict.** Rows A–D are 0.6 to 1.0 above the paper's; every GAN row of every seed, with or without resampling, is at or below row A. The paper's ZTF claim is not reproduced.

**Data.** 43,417 curves against the paper's 41,919: every class is 1–4% above Table 2 except CEP (181 against 182). The ±2σ class-range filter is our choice; it dropped 766 curves, and 719 more had fewer than 100 points. ZTF: 56,135 curves of 40 points; the counts equal Table 2 by construction except EB (31,183 against 31,477); 94 validation and 94 test curves per class.

**Training behaviour** (both GANs, checkpoints every 10k iterations).
- Validation TSTR accuracy is highest at the **first** checkpoint: 0.838 for the γ-GAN and 0.809 for the u-GAN at 10k iterations. For the next 360k iterations it stays between 0.74 and 0.79. This is the paper's Fig. 3 pattern (an early peak, then a fall), with the peak earlier than we expected.
- A caution on that peak: the validation set has 216 curves, and each point is 3 runs of an ensemble, so one point is uncertain by about ±0.03. The fall from 0.84 to 0.78 is about two such steps, not twenty.
- FID does not show the fall. For the u-GAN it keeps improving, from 1.14 at 10k to about 0.5 after 250k, while validation accuracy goes down. This is the paper's claim that FID misses the over-fitting.
- Recall (diversity) does show it. γ-GAN: macro recall 0.63–0.68 up to 40k iterations, 0.40–0.44 from 60k on. u-GAN: 0.61 at 10k, already 0.40–0.53 at 20–40k, 0.40–0.45 after. So the γ-GAN keeps its diversity about 30k iterations longer, which is the direction of the paper's Fig. 4. Density (fidelity) stays near 0.85–1.1 for both.
- The two extra γ-GAN seeds of session 2 repeat the pattern up to their last checkpoint (370k and 360k): validation accuracy 0.823 and 0.813 at 10k, 0.75–0.80 afterwards; macro recall 0.65–0.68 up to 30k and 0.40–0.44 from 60k on.
- **The rare classes lose their recall, and they lose all of it** (`per_class_density_recall.png`, γ-GAN, seed 0). CEP falls first (0.61 at 10k, 0.28 at 40k), then LPV and DSCT between 40k and 60k (from about 0.6 to below 0.1). From 100k on all three are at zero for the rest of the run, while the three common classes rise to 0.85–0.89. So the macro recall of 0.43 after 60k is 0.87 for three classes and zero for the other three. On ZTF the same happens to DSCT and CEP (0.6–0.7 at 10–20k, below 0.15 by 60k, zero from about 200k); LPV, a common class there with 5,057 training curves, keeps its recall. In both datasets the classes that collapse are those with a few hundred training curves (at most 423); the classes with thousands do not.
- What a recall of zero means here. The generated set is balanced, so each of the 109 CEP training conditions is used hundreds of times, and recall asks whether a real curve lies within the distance from a generated curve to its 5th nearest generated neighbour. It goes to zero when the generator returns almost the same curve every time for the same condition, whatever the noise vector. The density of these classes stays near 1: the curves are still realistic, they have stopped varying.
- The ZTF sample figure shows what they turn into. At the u-GAN's G-score checkpoint (310k) the generated DSCT and CEP curves lie on the training curve that supplied the condition, point for point. For the rare classes the late generator has memorised the training set. A classifier trained on such samples is trained on copies of the real curves, which is what row A does; and on ZTF the u-GAN rows, taken at 250k and 310k, do score like row A (73.3 and 73.4 against 73.2).
- This explains the early peak. The GAN helps while it still varies the rare classes, at 10–20k iterations, and stops helping once it copies them. It is the paper's over-fitting argument, seen directly, and it is the weakness ASTRA tests diffusion and flow matching against.

**Model selection.** Validation accuracy picked checkpoint 10k for both GANs. The G-score picked 10k for the u-GAN and 20k for the γ-GAN.
- On the density-recall plane (macro values printed in the log, checkpoints 10k–390k) each choice of the G-score is the single best point: the u-GAN's 10k has the highest density (1.11) and the highest recall (0.61) of its run, and the γ-GAN's 20k has the highest density (1.09) and the highest recall (0.68) of its run. So the G-score did what it is built to do.
- For the γ-GAN the classifier disagrees with it: validation TSTR accuracy is 0.838 at 10k and 0.793 at 20k, and the test accuracy follows (76.33 against 74.60). The checkpoint with the best fidelity and diversity is not the one that trains the best classifier here.
- The same will probably happen for seeds 1 and 2: in both, 20k again has the highest recall (0.68) with a density of 1.08 that no checkpoint of similar recall reaches, and a lower validation accuracy than 10k (0.793 and 0.796 against 0.823 and 0.813). Session 3 confirmed it: the G-score picked 20k for both, and its row is the lower one in all three seeds (by 1.7, 0.7 and 2.2 points). On the plane (`density_recall_plane.png`) the G-score's star sits on the top-right point of every run, so the criterion does find the checkpoint with the best fidelity and diversity. For γ = 0 both criteria picked 50k.
- On ZTF the validation accuracy gives no guidance: it stays between 71% and 74% from 20k to 400k with bands of about ±1.5, so its choices (250k for the u-GAN, 70k for the γ-GAN) are draws from noise. The G-score picked 310k and 30k.
- A caution: validation accuracy is measured on 216 curves, so its choice of 10k over 20k rests on a difference of 0.045, about one and a half times the ±0.03 above. That 10k is ahead in all three seeds is the stronger evidence.

**Rare classes.** Row A's per-class recall: EBSD/D 83.3, RRL 82.4, EBC 91.7, LPV 96.3, **DSCT 63.5, CEP 24.3**. The two rarest classes are the two worst, and CEP is close to a coin flip between a few classes. Soft warping moves CEP to 26.7–28.1. With 36 test curves per class, one curve is 2.8 points, so these differences are one or two curves. With GAN samples (rows E / F / G / H): **CEP 34.8 / 35.0 / 42.2 / 32.0** and **DSCT 69.1 / 69.4 / 68.1 / 65.4**. So the rarest class gains 8–18 points (3 to 6 of its 36 test curves) and DSCT gains 2–6 points (1 or 2 curves). The γ-GAN chosen by validation accuracy gives the largest CEP gain. The common classes pay a little for it: EBSD/D falls from 83.3 to 79.6 and RRL from 82.4 to 78.3 in row G. This is the question ASTRA is about, and the GAN's answer on Catalina is: yes for CEP, a little for DSCT.

**Over/under-fitting.** The classifiers of rows A–D have their lowest validation loss after 2.2–2.6 epochs on average and are stopped there; with balanced batches the rarest class is repeated about 66 times per epoch, so longer training over-fits it. The feature CNN stopped after 6 epochs (validation accuracy 0.741, test accuracy of this single CNN 0.731). The classifiers trained on GAN samples stop later: their best epoch is 4.0–4.2 on average for rows E, F and H and 6.5 for row G. In `trtr_loss_curves.png` the validation loss of rows A–D is lowest near epoch 2 and rises steadily afterwards, while the training loss keeps falling to 0.15–0.2: the classifier over-fits the oversampled real set within three epochs, on Catalina and on ZTF alike. Early stopping is what keeps rows A–D where they are.

**Mixed precision.** The critic's AMP scale stayed between 2,048 and 8,192 for the whole of both sessions, with no non-finite loss. Speed 10.9–12.4 iterations per second; one GAN job needs 10.4–10.6 hours, so it takes one session and the first hour of the next.

**Resuming.** Session 2 restored 105 files of session 1, and both paused GANs continued from their saved state (their next checkpoints, 380k and 370k, appeared within 12 minutes) and finished. This is the first check on a GPU that a paused job resumes.

**Deviations from the paper and their possible effect.** Rows A–D agree with the paper, so the classifier choices (batch 128, dropout 0.5, early stopping on the validation loss, mixed precision) and the pre-processing choices (±2σ filter, amplitude as half the peak-to-peak range, the warping interval) do not move the reference rows by more than their ±. With the GAN-side choices (batch 64, λ = 10, mixed precision, 3 selection runs per checkpoint instead of 15) rows E, F and G also agree with the paper. Row H does not, and two of these choices may play a part in that: with 3 selection runs instead of 15 the validation curve is noisier, and the paper does not say at which iterations its checkpoints were taken, so its G-score may have chosen among different candidates. Neither has been tested.